# แบบฝึกหัดบทที่ 3 (+ บทที่ 2) — ระบบผู้แนะนำอิงเนื้อหาบนชุดข้อมูลใหม่ **Goodbooks-10k**

สมุดบันทึกนี้ให้ผู้เรียน **เขียนโค้ดเองบางส่วน** โดยอาศัย Source Code ต้นแบบ (`cbrec_core.py` และ Lab 3-1 … 3-3)
บนชุดข้อมูลที่ *คล้ายกับ MovieLens แต่เป็นหนังสือ*: **Goodbooks-10k** (หนังสือ 10,000 เล่ม · คะแนน 1–5 จากผู้อ่าน ·
ชื่อเรื่อง · ผู้แต่ง · ภาษา · ปีพิมพ์ · tag ที่ผู้อ่านติดพร้อมจำนวนครั้ง) เรียงจากง่ายไปยาก แล้วปิดท้ายด้วยการ
**สร้างหลายโมเดลและวัดประสิทธิภาพเทียบกันตามบทที่ 2**

| ระดับ | ข้อ | เรื่อง | เชื่อมกับบท |
|---|---|---|---|
| ★ | E1–E2 | สร้าง "เนื้อหา" ของหนังสือ · TF-IDF · คำนวณ IDF เอง | 3.4–3.6 |
| ★★ | E3–E4 | แบ่งข้อมูลเมื่อ **ไม่มีเวลากำกับ** · ระบบฐาน TopPopular/Random | 2.8 · 2.2 |
| ★★★ | E5–E7 | โปรไฟล์ centroid/Rocchio · BM25 (ปรับ k₁, b บน validation) · การผสาน RRF | 3.8 · 3.11 · 3.18 |
| ★★★★ | E8–E10 | candidate set full vs sampled · guardrail + MMR · ทดสอบครั้งเดียว + bootstrap/Holm | 2.9 · 2.5 · 2.10 |
| ☆ | E11 | อ่านผลรายคน (ประวัติ · ของจริง · Top-K พร้อมเนื้อหา) · ทำนายคะแนน MAE/RMSE | 2 · 3 |
| ☆ | Quiz | คำถามแนวคิด 10 ข้อ ตรวจอัตโนมัติ | 2 · 3 |

### วิธีทำ
1. รันส่วน 0–2 (ติดตั้ง · แกนกลาง · โหลดข้อมูล) ให้ครบก่อน
2. ทำทีละข้อ: อ่านโจทย์ → เติมโค้ดตรง `TODO` → รันเซลล์ของข้อนั้น → รันเซลล์ **ตรวจคำตอบ**
3. ถ้าไม่ผ่าน แก้แล้วรันใหม่ได้ไม่จำกัด · เมื่อครบให้รันเซลล์ **สรุปคะแนน** ท้ายเล่ม
4. นำตัวเลข/ตารางที่ได้ไปตอบ **คำถามเขียนอธิบาย** ในไฟล์ Word `Exercise3_TextOnly_คำถามเขียนอธิบาย.docx`

> ทุกข้อใช้ฟังก์ชันจาก `cbrec_core.py` ที่ฝังมาในเล่ม — ไม่ต้องเขียนอัลกอริทึมใหม่ แต่ต้อง **เลือกใช้ให้ถูกและตีความให้เป็น**

> ## ฉบับผู้เรียน
> เติมโค้ดเฉพาะบรรทัดที่มี `TODO` · ถ้ายังไม่เติม เซลล์ตรวจจะแจ้งว่า "ยังไม่ได้ทำ" (ไม่ใช่ error)

## ส่วน 0 — ติดตั้งและตั้งค่า

In [ ]:
import importlib, subprocess, sys, os, urllib.request
for pkg in ["ipywidgets"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 180); pd.set_option("display.max_colwidth", 70)

SEED = 2569; K = 10; LIKE_THR = 4.0; DISLIKE_THR = 2.0
N_USERS_SAMPLE = 1500      # สุ่มผู้ใช้จาก 53,424 คน เพื่อให้รันได้ใน 2–3 นาที (ประกาศ)
MIN_RATINGS_USER = 30      # เลือกเฉพาะผู้ใช้ที่ให้คะแนน ≥ 30 เล่ม
DATA_DIR = "data/goodbooks-10k"


## ส่วน 1 — แกนกลาง `cbrec_core` (ฝังทั้งไฟล์ — ไฟล์เดียวกับ Lab 3-1 … 3-3)

In [ ]:
# -*- coding: utf-8 -*-
"""
==========================================================================================
 cbrec_core.py — แกนกลางของระบบผู้แนะนำอิงเนื้อหา (Content-Based) สำหรับบทที่ 3
                 + ตัวชี้วัดการประเมินผล (Accuracy / Beyond-Accuracy) ของบทที่ 2
                 + เครื่องมือออกแบบการทดลองแบบออฟไลน์ (Offline Experiment)
==========================================================================================
ไฟล์นี้ถูก "ฝัง" อยู่ในสมุดบันทึกทุกเล่ม (Lab3_1 / Lab3_2 / Lab3_3) เพื่อให้แต่ละเล่มรันบน
Google Colab ได้โดยอัปโหลดไฟล์เดียว และยังบันทึกแยกเป็น cbrec_core.py เพื่อ import ซ้ำได้

โครงสร้าง (เรียงตามลำดับของบทที่ 3 → บทที่ 2)
  ส่วนที่ 1  การเตรียมข้อความ (3.4–3.5)   : tokenize · คำหยุด · n-gram
  ส่วนที่ 2  การถ่วงน้ำหนักคำ (3.6–3.7)   : TF 5 รูปแบบ · IDF หลายสูตร · L2 · โคไซน์
  ส่วนที่ 3  BM25 / BM25+ (3.11)          : การอิ่มตัวและปรับความยาวเอกสาร
  ส่วนที่ 4  ปริภูมิความหมาย (3.13)         : LSA (TruncatedSVD) · Sentence-Embedding (ทางเลือก)
  ส่วนที่ 5  โปรไฟล์ผู้ใช้ (3.8, 3.12)      : centroid · ถ่วงคะแนน · Rocchio · Naive Bayes · Logistic
  ส่วนที่ 6  ฟังก์ชันจับคู่และจัดอันดับ (3.9–3.10) : โคไซน์ · Item-kNN · Top-K · MMR
  ส่วนที่ 7  ผสานคะแนนและปรับเทียบ (3.18)  : convex · RRF · Platt · ECE
  ส่วนที่ 8  โมเดลทำนายคะแนน (Rating)      : ค่าเฉลี่ย · Content-kNN regression
  ส่วนที่ 9  ตัวชี้วัดบทที่ 2               : MAE/RMSE · P/R/F1/HR@K · MRR · MAP · nDCG · AUC
                                            Coverage · Gini · LongTail · ILD · Novelty · Calibration · Fairness
  ส่วนที่ 10 การออกแบบการทดลองออฟไลน์ (2.7–2.10, 2.14) : split 4 แบบ · leakage check · k-core ·
                                            candidate set · seed variance · paired bootstrap · Holm · ProtocolCard
  ส่วนที่ 11 การนำเสนอผลและ GUI           : ตารางเทียบโมเดล · กราฟ · รายงานผู้ใช้ · ฟอร์ม ipywidgets

หมายเหตุด้านการทำซ้ำ (Reproducibility)
  * ทุกฟังก์ชันที่มีการสุ่มรับพารามิเตอร์ seed และใช้ numpy.random.default_rng(seed)
  * สูตรตั้งต้นทุกตัวตรงกับที่ใช้คำนวณตัวเลขในบทที่ 3 (TF = 1+log2 f, IDF = ln((1+N)/(1+df))+1, L2)
    หากเปลี่ยนสูตร ตัวเลขจะเปลี่ยนทันที — ต้อง "ประกาศการตั้งค่า" ทุกครั้งที่รายงานผล (ตาราง 3.3)
"""
import math
import re
import time
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings("ignore")

SEED = 2569  # ค่าตั้งต้นของ seed ทั้งชุด (พ.ศ. 2569) — เปลี่ยนแล้วผลเปลี่ยน แต่ข้อสรุปเชิงแนวคิดควรคงเดิม


def set_seed(seed=SEED):
    """ตั้ง seed ของ numpy และ random (และ torch หากมี) ให้รันซ้ำได้ผลเดิม"""
    import random
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
    except Exception:
        pass
    return np.random.default_rng(seed)


class Timer:
    """ตัวจับเวลาอย่างง่าย: with Timer('ชื่อ'): ...  → พิมพ์เวลาที่ใช้ (ใช้ดูต้นทุนของแต่ละขั้น)"""
    def __init__(self, label=""):
        self.label = label
    def __enter__(self):
        self.t0 = time.time(); return self
    def __exit__(self, *a):
        self.sec = time.time() - self.t0
        print(f"[เวลา] {self.label}: {self.sec:.2f} วินาที")


# =========================================================================================
# ส่วนที่ 1  การเตรียมข้อความ (หัวข้อ 3.4–3.5)
# =========================================================================================
# รายการคำหยุดชุดเดียวกับที่ใช้คำนวณตัวเลขทุกตัวในบทที่ 3 (ตาราง 3.3: 457 → 263 โทเคน, 249 → 203 คำ)
# ข้อสังเกต: รายการนี้ "ตั้งใจ" เก็บคำปฏิเสธบางคำ (เช่น not, no) ไว้ในรายการคำหยุดตามต้นฉบับของบท
#           หากนำไปใช้กับงานวิเคราะห์ความรู้สึก ต้องนำ not/no/never ออกจากรายการ (หัวข้อ 3.4.6)
STOP_EN = set("""
a an the and or of to in on at for with from as is are was were be been being am
his her their its it he she they them him us we our you your i my me
who which that this these those what when where why how whom whose
not no nor but by into across over under out up down off above below through
while after before during until since again once here there
must can could may might will would shall should do does did have has had
so such than then too very just only own each every all both few more most other same
one two another between within
""".split())


def tokenize(text, keep_thai=False):
    """ตัดคำแบบง่ายที่สุด: แปลงเป็นตัวพิมพ์เล็กแล้วเก็บเฉพาะสายอักษรละติน [a-z]+
    (ตรงกับกฎที่ใช้ในบทที่ 3 ทำให้ "world." → world, "job:" → job, "Mad Max: Fury Road" → mad max fury road)
    keep_thai=True จะเก็บกลุ่มอักขระไทยไว้ด้วย (ยังไม่ตัดคำไทย — ต้องใช้ pythainlp ในหัวข้อ 3.5)
    """
    text = re.sub(r"'s\b", "", text.lower())     # ตัด 's แสดงความเป็นเจ้าของ (potter's → potter) กันโทเคน "s" ลอย
    if keep_thai:
        return re.findall(r"[a-z]+|[฀-๿]+", text)
    return re.findall(r"[a-z]+", text)


def preprocess(text, stop=STOP_EN, ngram_range=(1, 1), keep_thai=False, stemmer=None):
    """กระบวนงานเตรียมข้อความ (ภาพที่ 3.5): ตัดคำ → ตัดคำหยุด → (ลดรูปคำ) → (เพิ่ม n-gram)
    คืนรายการโทเคน (หน่วยที่จะกลายเป็น "มิติ" ของเวกเตอร์)
    * stemmer: ฟังก์ชัน str→str (เช่น nltk PorterStemmer().stem) ค่าตั้งต้น None = ไม่ลดรูปคำ
    * ngram_range=(1,2) จะเพิ่ม bigram ในรูป "w1_w2" ต่อท้าย unigram (ตาราง 3.11)
    """
    toks = [t for t in tokenize(text, keep_thai) if t not in stop]
    if stemmer is not None:
        toks = [stemmer(t) for t in toks]
    lo, hi = ngram_range
    out = []
    for n in range(lo, hi + 1):
        if n == 1:
            out.extend(toks)
        else:
            out.extend("_".join(toks[i:i + n]) for i in range(len(toks) - n + 1))
    return out


def corpus_stats(token_docs):
    """สถิติของคลัง (ตาราง 3.3): จำนวนเอกสาร โทเคนรวม ขนาดคลังศัพท์ ความยาวเฉลี่ย avgdl"""
    n_tok = sum(len(d) for d in token_docs)
    vocab = set(t for d in token_docs for t in d)
    return {"N (เอกสาร)": len(token_docs), "โทเคนรวม": n_tok, "ขนาดคลังศัพท์": len(vocab),
            "avgdl": n_tok / max(1, len(token_docs))}


# =========================================================================================
# ส่วนที่ 2  การถ่วงน้ำหนักคำ TF / IDF / L2 และโคไซน์ (หัวข้อ 3.6–3.7)
# =========================================================================================
def tf_weight(f, scheme="log", max_f=1, doc_len=1):
    """TF ห้ารูปแบบ (ตาราง 3.20)
    raw    : f                      นับตรง ๆ (คำที่ซ้ำ 3 ครั้งได้ 3 เท่า — ไม่มีเพดาน)
    log    : 1 + log2(f)            ลดทอนแบบลอการิทึม (ค่าตั้งต้นของบท; sublinear_tf)
    binary : 1 ถ้า f>0              สนใจแค่ "มี/ไม่มี"
    aug    : 0.5 + 0.5 f/max_f      ปรับตามคำที่ถี่ที่สุดในเอกสาร (กันเอกสารยาวได้เปรียบ)
    norm   : f / doc_len            สัดส่วนต่อความยาวเอกสาร
    """
    if f <= 0:
        return 0.0
    if scheme == "raw":
        return float(f)
    if scheme == "log":
        return 1.0 + math.log2(f)
    if scheme == "binary":
        return 1.0
    if scheme == "aug":
        return 0.5 + 0.5 * f / max(1, max_f)
    if scheme == "norm":
        return f / max(1, doc_len)
    raise ValueError(scheme)


def idf_weight(df, N, scheme="smooth"):
    """IDF หลายสูตร (ตาราง 3.24 — TF-IDF เป็น "ตระกูล" ของสูตร ไม่ใช่สูตรเดียว)
    smooth   : ln((1+N)/(1+df)) + 1   ค่าตั้งต้นของบท (เหมือน sklearn smooth_idf) — ไม่มีคำใดได้ 0
    standard : ln(N/df)               สูตรดั้งเดิม คำที่พบทุกเอกสารได้ 0 พอดี
    prob     : max(0, ln((N-df)/df))  แบบความน่าจะเป็น
    bm25     : ln((N-df+0.5)/(df+0.5) + 1)  สูตรที่ BM25 (Lucene) ใช้
    none     : 1                      ไม่ถ่วง IDF (เท่ากับ TF อย่างเดียว)
    """
    if scheme == "smooth":
        return math.log((1 + N) / (1 + df)) + 1.0
    if scheme == "standard":
        return math.log(N / df)
    if scheme == "prob":
        return max(0.0, math.log((N - df) / df)) if df < N else 0.0
    if scheme == "bm25":
        return math.log((N - df + 0.5) / (df + 0.5) + 1.0)
    if scheme == "none":
        return 1.0
    raise ValueError(scheme)


class TextVectorizer:
    """เวกเตอร์ไรเซอร์ที่เขียนเองเพื่อให้ "เห็นทุกขั้น" (ต่างจาก sklearn ตรงที่เลือกสูตร TF/IDF ได้อิสระ)

    พารามิเตอร์ (ตาราง 3.67)
      tf, idf        : รูปแบบ TF และ IDF (ดู tf_weight / idf_weight)
      min_df, max_df : ตัดคำที่พบน้อยเกิน (จำนวนเอกสาร) / บ่อยเกิน (สัดส่วนเอกสาร) — คุมขนาดคลังศัพท์
      ngram_range    : (1,1) unigram · (1,2) unigram+bigram
      norm           : 'l2' (ค่าตั้งต้น — ทำให้ผลคูณจุด = โคไซน์) หรือ None
      max_features   : เก็บเฉพาะคำที่ df สูงสุด N คำ (None = ทั้งหมด)
    หลัง fit จะมี  vocab_ (list), idx_ (dict คำ→คอลัมน์), df_ (Counter), idf_ (np.array),
                   doc_len_ (ความยาวเอกสารฝึกหลังเตรียมข้อความ), avgdl_
    ข้อสังเกต: IDF และ df คำนวณจาก "เอกสารที่ fit" เท่านั้น — ในการทดลองต้อง fit ด้วยคลังรายการ
               (ซึ่งไม่ใช่ข้อมูลผู้ใช้จึงไม่ถือว่ารั่ว) แต่สถิติของผู้ใช้ทุกตัวต้องมาจากชุดฝึก (ตาราง 2.10)
    """
    def __init__(self, tf="log", idf="smooth", min_df=1, max_df=1.0, ngram_range=(1, 1),
                 stop=STOP_EN, norm="l2", max_features=None, keep_thai=False, stemmer=None):
        self.tf, self.idf, self.min_df, self.max_df = tf, idf, min_df, max_df
        self.ngram_range, self.stop, self.norm = ngram_range, stop, norm
        self.max_features, self.keep_thai, self.stemmer = max_features, keep_thai, stemmer

    def _tokens(self, docs):
        return [preprocess(d, self.stop, self.ngram_range, self.keep_thai, self.stemmer) for d in docs]

    def fit(self, docs):
        toks = self._tokens(docs)
        N = len(toks)
        df = Counter()
        for d in toks:
            df.update(set(d))
        max_df_abs = self.max_df * N if isinstance(self.max_df, float) else self.max_df
        vocab = [t for t, c in df.items() if c >= self.min_df and c <= max_df_abs]
        if self.max_features:
            vocab = sorted(vocab, key=lambda t: (-df[t], t))[: self.max_features]
        self.vocab_ = sorted(vocab)                 # เรียงตามตัวอักษรเพื่อให้ตรวจสอบด้วยมือได้ (ตาราง 3.13)
        self.idx_ = {t: k for k, t in enumerate(self.vocab_)}
        self.df_ = df
        self.N_ = N
        self.idf_ = np.array([idf_weight(df[t], N, self.idf) for t in self.vocab_])
        self.doc_len_ = np.array([len(d) for d in toks])
        self.avgdl_ = float(self.doc_len_.mean()) if N else 0.0
        self._train_tokens = toks
        return self

    def transform(self, docs, tokens=None):
        """คืน scipy.sparse CSR ขนาด (จำนวนเอกสาร × ขนาดคลังศัพท์)"""
        toks = tokens if tokens is not None else self._tokens(docs)
        rows, cols, vals = [], [], []
        for r, d in enumerate(toks):
            c = Counter(t for t in d if t in self.idx_)
            if not c:
                continue
            max_f = max(c.values())
            for t, f in c.items():
                j = self.idx_[t]
                rows.append(r); cols.append(j)
                vals.append(tf_weight(f, self.tf, max_f, len(d)) * self.idf_[j])
        X = sp.csr_matrix((vals, (rows, cols)), shape=(len(toks), len(self.vocab_)), dtype=np.float64)
        if self.norm == "l2":
            X = l2_normalize_rows(X)
        return X

    def fit_transform(self, docs):
        self.fit(docs)
        return self.transform(docs, tokens=self._train_tokens)

    def top_terms(self, vec, k=10):
        """คำที่มีน้ำหนักสูงสุด k คำของเวกเตอร์หนึ่งแถว (ใช้ "อ่าน" โปรไฟล์หรือเอกสาร — หัวข้อ 3.9.4)"""
        v = np.asarray(vec.todense()).ravel() if sp.issparse(vec) else np.asarray(vec).ravel()
        order = np.argsort(-v)[:k]
        return [(self.vocab_[j], round(float(v[j]), 4)) for j in order if v[j] > 0]


def l2_normalize_rows(X):
    """ปรับความยาวแถวเป็น 1 (L2) — เอกลักษณ์ 3.7.2: หลัง L2 แล้ว ผลคูณจุด = โคไซน์ และ ‖a−b‖² = 2(1−cos)
    แถวที่เป็นศูนย์ทั้งแถว (เอกสารไม่มีคำในคลังศัพท์) จะคงเป็นศูนย์ ไม่หารด้วยศูนย์ (ตาราง 3.68 — NaN)"""
    if sp.issparse(X):
        X = sp.csr_matrix(X, dtype=np.float64)
        n = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel())
        n[n == 0] = 1.0
        return sp.diags(1.0 / n) @ X
    X = np.asarray(X, dtype=np.float64)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return X / n


def cosine_matrix(X, Y=None):
    """เมทริกซ์โคไซน์ทุกคู่ (X: n×d, Y: m×d) — คืน dense n×m ; ถ้า X ผ่าน L2 แล้ว = X·Yᵀ"""
    Xn = l2_normalize_rows(X)
    Yn = Xn if Y is None else l2_normalize_rows(Y)
    S = Xn @ Yn.T
    return np.asarray(S.todense()) if sp.issparse(S) else np.asarray(S)


def to_dense_row(v):
    return np.asarray(v.todense()).ravel() if sp.issparse(v) else np.asarray(v, dtype=np.float64).ravel()


# =========================================================================================
# ส่วนที่ 3  BM25 / BM25+ (หัวข้อ 3.11)
# =========================================================================================
class BM25:
    """BM25 (Robertson) พร้อมส่วนขยาย BM25+ (delta) และเลือก IDF ได้
        score(q, d) = Σ_{t∈q} idf(t) · [ f(t,d)(k1+1) / (f(t,d) + k1(1 − b + b·|d|/avgdl)) + delta ]
    k1 คุมความอิ่มตัวของความถี่ (0.5–3.0 ค่าตั้งต้น 1.5), b คุมการปรับความยาว (0–1 ค่าตั้งต้น 0.75)
    delta>0 คือ BM25+ (ป้องกันเอกสารยาวถูกลดคะแนนมากเกินควร — หัวข้อ 3.11.4)
    ตัวเลขในตาราง 3.46 ใช้ k1=1.5, b=0.75, idf แบบ bm25: ln((N−df+0.5)/(df+0.5)+1)
    """
    def __init__(self, k1=1.5, b=0.75, delta=0.0, idf="bm25"):
        self.k1, self.b, self.delta, self.idf_scheme = k1, b, delta, idf

    def fit(self, token_docs):
        self.docs = token_docs
        self.N = len(token_docs)
        self.doc_len = np.array([len(d) for d in token_docs], dtype=np.float64)
        self.avgdl = self.doc_len.mean() if self.N else 0.0
        self.df = Counter()
        for d in token_docs:
            self.df.update(set(d))
        self.tf = [Counter(d) for d in token_docs]
        self.idf = {t: idf_weight(c, self.N, self.idf_scheme) for t, c in self.df.items()}
        # เมทริกซ์ความถี่คำแบบเบาบาง (N × V) เก็บเป็น CSC เพื่อดึง "คอลัมน์ของคำ" ได้เร็ว — ทำให้ให้คะแนนข้อคำถามยาว ๆ ได้ทันที
        self.vocab_ = sorted(self.df)
        self.idx_ = {t: j for j, t in enumerate(self.vocab_)}
        rows, cols, vals = [], [], []
        for r, c in enumerate(self.tf):
            for t, f in c.items():
                rows.append(r); cols.append(self.idx_[t]); vals.append(f)
        self.TF_ = sp.csc_matrix((vals, (rows, cols)), shape=(self.N, len(self.vocab_)), dtype=np.float64)
        self.K_ = self.k1 * (1 - self.b + self.b * self.doc_len / max(self.avgdl, 1e-9))   # ตัวหารส่วนปรับความยาว ต่อเอกสาร
        return self

    def score_query(self, query_terms, term_weights=None):
        """คะแนน BM25 ของ "ทุกเอกสาร" ต่อข้อคำถามหนึ่งชุด (query_terms = รายการคำ; น้ำหนักคำใช้ได้)
        คืน np.array ยาว N — เอกสารที่ไม่มีคำในข้อคำถามเลยจะได้ 0 พอดี (ต่างจากโคไซน์ที่ได้ค่าบวกเล็ก ๆ จากคำอื่น)"""
        s = np.zeros(self.N)
        for t in query_terms:
            j = self.idx_.get(t)
            if j is None:
                continue
            w = 1.0 if term_weights is None else term_weights.get(t, 1.0)
            col = self.TF_[:, j]                                   # เฉพาะเอกสารที่มีคำ t (เบาบาง)
            docs, f = col.indices, col.data
            comp = f * (self.k1 + 1) / (f + self.K_[docs]) + self.delta
            s[docs] += w * self.idf[t] * comp
        return s

    def explain(self, query_terms, doc_index):
        """แจกแจงองค์ประกอบรายคำของเอกสารหนึ่งฉบับ (ใช้อ่านตาราง 3.46 ทีละคอลัมน์)"""
        rows = []
        K = self.k1 * (1 - self.b + self.b * self.doc_len[doc_index] / max(self.avgdl, 1e-9))
        for t in query_terms:
            f = self.tf[doc_index].get(t, 0)
            if f == 0:
                continue
            comp = f * (self.k1 + 1) / (f + K) + self.delta
            rows.append({"คำ": t, "f": f, "องค์ประกอบ TF": round(comp, 4), "idf": round(self.idf[t], 4),
                         "คะแนน": round(comp * self.idf[t], 4)})
        return pd.DataFrame(rows)


# =========================================================================================
# ส่วนที่ 4  ปริภูมิความหมาย: LSA และ Sentence Embedding (หัวข้อ 3.13)
# =========================================================================================
class LSA:
    """Latent Semantic Analysis = TruncatedSVD บนเมทริกซ์ TF-IDF แล้วปรับ L2
    k (มิติแฝง) ต้องน้อยกว่า min(จำนวนเอกสาร, ขนาดคลังศัพท์) — ในคลังเล็กอย่าง MovieMini-12 (N=12)
    ค่า k จึงเล็กมากและ "ไม่มีสถิติพอ" ให้เชื่อ (ข้อจำกัดที่บทตั้งใจให้เห็น หัวข้อ 3.13.2)"""
    def __init__(self, k=50, seed=SEED):
        self.k, self.seed = k, seed

    def fit(self, X):
        from sklearn.decomposition import TruncatedSVD
        k = int(min(self.k, min(X.shape) - 1))
        self.svd = TruncatedSVD(n_components=max(1, k), random_state=self.seed).fit(X)
        self.explained_ = self.svd.explained_variance_ratio_.sum()
        return self

    def transform(self, X):
        return l2_normalize_rows(self.svd.transform(X))

    def fit_transform(self, X):
        return self.fit(X).transform(X)


def sentence_embeddings(texts, model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=64):
    """เวกเตอร์ความหมายจาก Bi-Encoder สำเร็จรูป (ต้อง pip install sentence-transformers และมีอินเทอร์เน็ต)
    คืน np.array (n × d) ที่ปรับ L2 แล้ว ; หากโหลดไม่ได้จะคืน None ให้ผู้เรียกใช้ LSA แทน
    ข้อสังเกต: ผลลัพธ์ขึ้นกับรุ่นโมเดลและ "กลยุทธ์การรวมค่า" (mean pooling) ที่โมเดลนั้นใช้ (3.13.5)"""
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(model_name)
        E = m.encode(list(texts), batch_size=batch_size, show_progress_bar=False, normalize_embeddings=True)
        return np.asarray(E, dtype=np.float64)
    except Exception as e:  # ไม่มีไลบรารี / ไม่มีเน็ต
        print("[คำเตือน] ใช้ sentence-transformers ไม่ได้:", str(e)[:120], "→ ใช้ LSA แทน")
        return None


# =========================================================================================
# ส่วนที่ 5  โปรไฟล์ผู้ใช้ (หัวข้อ 3.8 และ 3.12)
# =========================================================================================
def build_profile(X, hist, method="centroid", like_thr=4, dislike_thr=2, beta=0.8, gamma=0.2,
                  alpha=0.0, query_vec=None, clip_negative=True, normalize=True):
    """สร้างเวกเตอร์โปรไฟล์ในปริภูมิเดียวกับรายการ (หลักการข้อเดียวของ 3.8.1)

    X      : เมทริกซ์รายการ (n_items × d) — sparse หรือ dense ; แถวควรผ่าน L2 แล้ว
    hist   : dict {item_index: rating} ประวัติของผู้ใช้ (ดัชนีแถวของ X)
    method :
      'centroid' : ค่าเฉลี่ยของเวกเตอร์รายการที่ชอบ (rating ≥ like_thr)                     (3.8.2)
      'weighted' : Σ (r−3)·v / Σ|r−3|  ใช้ทั้งชอบและไม่ชอบ ค่ากลาง (3) ไม่มีผล               (3.8.4)
      'rocchio'  : α·q + β·mean(ชอบ) − γ·mean(ไม่ชอบ)  ค่าตั้งต้น β=0.8, γ=0.2 (กฎ β>γ)       (3.8.5)
    clip_negative=True : ตัดมิติที่ติดลบเป็น 0 ก่อนปรับ L2 (ตามตัวอย่างในหัวข้อ 3.8.6)
    คืน np.array ยาว d (ปรับ L2 แล้วเมื่อ normalize=True) — ถ้าไม่มีรายการที่ชอบเลยคืนเวกเตอร์ศูนย์
    ข้อสังเกต: เวกเตอร์ศูนย์ = "ไม่มีหลักฐาน" ต้องแยกจากคะแนนต่ำ (3.10.2) ผู้เรียกควร fallback เป็นความนิยม
    """
    d = X.shape[1]
    liked = [i for i, r in hist.items() if r >= like_thr]
    disliked = [i for i, r in hist.items() if r <= dislike_thr]

    def mean_rows(idx):
        if not idx:
            return np.zeros(d)
        M = X[idx]
        return to_dense_row(M.mean(axis=0))

    if method == "centroid":
        p = mean_rows(liked)
    elif method == "weighted":
        num = np.zeros(d); den = 0.0
        for i, r in hist.items():
            w = r - 3.0
            if w == 0:
                continue
            num += w * to_dense_row(X[i]); den += abs(w)
        p = num / den if den > 0 else num
    elif method == "rocchio":
        q = np.zeros(d) if query_vec is None else to_dense_row(query_vec)
        p = alpha * q + beta * mean_rows(liked) - gamma * mean_rows(disliked)
    else:
        raise ValueError(method)
    if clip_negative:
        p = np.clip(p, 0, None)
    if normalize:
        n = np.linalg.norm(p)
        if n > 0:
            p = p / n
    return p


class ClassifierProfile:
    """โปรไฟล์แบบ "เรียนรู้ได้" รายผู้ใช้ (หัวข้อ 3.12): Multinomial Naive Bayes หรือ Logistic Regression
    มองการแนะนำเป็นการจำแนกสองคลาส: ชอบ (rating ≥ like_thr) กับ ไม่ชอบ (≤ dislike_thr)
      * NB ต้องใช้ "จำนวนนับ" (เมทริกซ์ที่ค่าไม่ติดลบ) และมีการปรับเรียบลาปลัส alpha (นิยาม 3.11)
      * LR เรียนน้ำหนักรายมิติโดยตรง ใช้ C (ส่วนกลับของ regularization) — ต้องการข้อมูลมากกว่า (ตาราง 3.47)
    ถ้าผู้ใช้มีตัวอย่างไม่ครบทั้งสองคลาส หรือน้อยกว่า min_examples จะ "ฝึกไม่ได้" → score() คืน None
    ให้ผู้เรียกใช้โปรไฟล์ centroid แทน (บันได 3.48: ระดับ 1–2 ก่อน ระดับ 3)
    """
    def __init__(self, kind="nb", alpha=1.0, C=1.0, like_thr=4, dislike_thr=2, min_examples=2, seed=SEED):
        self.kind, self.alpha, self.C = kind, alpha, C
        self.like_thr, self.dislike_thr, self.min_examples, self.seed = like_thr, dislike_thr, min_examples, seed
        self.model = None

    def fit(self, X, hist):
        pos = [i for i, r in hist.items() if r >= self.like_thr]
        neg = [i for i, r in hist.items() if r <= self.dislike_thr]
        if len(pos) < self.min_examples or len(neg) < 1:
            self.model = None
            return self
        idx = pos + neg
        y = np.array([1] * len(pos) + [0] * len(neg))
        Xs = X[idx]
        if self.kind == "nb":
            from sklearn.naive_bayes import MultinomialNB
            self.model = MultinomialNB(alpha=self.alpha).fit(Xs, y)
        else:
            from sklearn.linear_model import LogisticRegression
            self.model = LogisticRegression(C=self.C, max_iter=1000, random_state=self.seed).fit(Xs, y)
        return self

    def score(self, X):
        """ความน่าจะเป็น P(ชอบ | เนื้อหา) ของทุกรายการ หรือ None เมื่อฝึกไม่ได้"""
        if self.model is None:
            return None
        return self.model.predict_proba(X)[:, 1]


# =========================================================================================
# ส่วนที่ 6  ฟังก์ชันจับคู่ การจัดอันดับ Top-K และ MMR (หัวข้อ 3.9–3.10)
# =========================================================================================
def match_scores(X, profile):
    """คะแนนจับคู่ = ผลคูณจุดระหว่างทุกแถวของ X กับโปรไฟล์ (เมื่อทั้งคู่ผ่าน L2 = โคไซน์) คืน np.array ยาว n_items"""
    s = X @ profile
    return np.asarray(s).ravel()


def item_knn_scores(S, liked_idx, agg="mean", k=None):
    """Item-kNN เชิงเนื้อหา (3.9.5): เทียบผู้สมัครกับรายการที่ชอบ "ทีละรายการ" แล้วรวม
    S: เมทริกซ์ความคล้าย n×n ; agg='mean' เฉลี่ย, 'max' เอาค่าสูงสุด ; k = ใช้เฉพาะ k เพื่อนบ้านที่คล้ายที่สุด"""
    if not liked_idx:
        return np.zeros(S.shape[0])
    sims = S[:, liked_idx]                       # n × |liked|
    if k is not None and k < sims.shape[1]:
        sims = -np.sort(-sims, axis=1)[:, :k]
    return sims.mean(axis=1) if agg == "mean" else sims.max(axis=1)


def explain_terms(vectorizer, profile, item_vec, k=8):
    """อธิบายผล (3.9.4): คำใดทำให้คะแนนสูง = ส่วนร่วมรายมิติ p_j · v_j เรียงจากมากไปน้อย"""
    v = to_dense_row(item_vec)
    contrib = profile * v
    order = np.argsort(-contrib)[:k]
    return [(vectorizer.vocab_[j], round(float(profile[j]), 4), round(float(v[j]), 4), round(float(contrib[j]), 4))
            for j in order if contrib[j] > 0]


def top_k(scores, seen, K=10, min_score=None, item_ids=None, candidates=None):
    """ห้าขั้นของ 3.10.1: รวบรวมผู้สมัคร → (คะแนนมีแล้ว) → กรองที่เคยเห็น → เรียง+ตัดสินเสมอ → เกณฑ์ขั้นต่ำ → ตัด K
    seen       : เซตดัชนีรายการที่ผู้ใช้เคยมีปฏิสัมพันธ์ในชุดฝึก (ห้ามแนะนำซ้ำ)
    candidates : เซตดัชนีผู้สมัคร (None = ทั้งคลัง = full-catalog ranking นิยาม 2.7)
    ตัดสินเสมอ : คะแนนเท่ากันเรียงตามดัชนี/รหัสรายการ (กฎที่ทำซ้ำได้ของบท — 3.10.2) ผ่าน np.lexsort
    คืน list ดัชนี (ยาว ≤ K)"""
    n = len(scores)
    cand = np.arange(n) if candidates is None else np.asarray(sorted(candidates))
    if seen:
        cand = cand[~np.isin(cand, list(seen))]
    if min_score is not None:
        cand = cand[scores[cand] >= min_score]
    if len(cand) == 0:
        return []
    key2 = cand if item_ids is None else np.asarray([item_ids[i] for i in cand])
    order = np.lexsort((key2, -scores[cand]))     # เรียงคะแนนมาก→น้อย แล้วรหัสน้อย→มากเมื่อเสมอ
    return [int(i) for i in cand[order][:K]]


def mmr_rerank(scores, S, K=10, lam=0.7, seen=None, pool=None, min_score=None, normalize=True):
    """Maximal Marginal Relevance (นิยาม 3.8): เลือกทีละรายการแบบละโมบ
        argmax_i  λ·rel(i) − (1−λ)·max_{j∈เลือกแล้ว} sim(i, j)
    λ=1 → เรียงตามความเกี่ยวข้องล้วน ; λ→0 → เน้นความหลากหลาย ; ค่านิยม 0.7 (ตัวอย่าง 3.9)
    pool: จำนวนผู้สมัครต้นทางที่นำมาจัดอันดับซ้ำ (ค่าตั้งต้น = 5K) เพื่อประหยัดเวลาในคลังใหญ่
    normalize=True จะปรับ rel ด้วย min-max ให้อยู่ในมาตรา 0–1 เดียวกับ sim (จำเป็นเมื่อคะแนนเป็น BM25/RRF)
    normalize=False ใช้คะแนนดิบ (ตรงกับตัวอย่างการคำนวณ 3.9 ในตำรา ซึ่ง rel อยู่ในช่วง 0–1 อยู่แล้ว)"""
    pool = pool or 5 * K
    base = top_k(scores, seen or set(), K=pool, min_score=min_score)
    if not base:
        return []
    rel = scores[base]
    if normalize and rel.max() > rel.min():
        rel = (rel - rel.min()) / (rel.max() - rel.min())
    chosen, remaining = [], list(range(len(base)))
    while remaining and len(chosen) < K:
        best, best_val = None, -1e18
        for r in remaining:
            div = max(S[base[r], base[c]] for c in chosen) if chosen else 0.0
            val = lam * rel[r] - (1 - lam) * div
            if val > best_val:
                best, best_val = r, val
        chosen.append(best); remaining.remove(best)
    return [base[c] for c in chosen]


# =========================================================================================
# ส่วนที่ 7  ผสานคะแนน · ปรับเทียบ (หัวข้อ 3.18)
# =========================================================================================
def minmax(x):
    """ปรับมาตรา 0–1 (ข้อบังคับก่อนผสมแบบนูน — ภาพที่ 3.32) ; ถ้าค่าคงที่คืนศูนย์ทั้งหมด"""
    x = np.asarray(x, dtype=np.float64)
    lo, hi = np.nanmin(x), np.nanmax(x)
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)


def zscore(x):
    x = np.asarray(x, dtype=np.float64)
    s = x.std()
    return (x - x.mean()) / s if s > 0 else np.zeros_like(x)


def convex_fusion(score_list, weights=None, scale="minmax"):
    """ผสมแบบนูน: Σ w_k · scale(s_k) โดย Σw = 1 (ตัวอย่าง 3.22) — ผู้ชนะเปลี่ยนตาม w จึงต้องปรับ w บนชุดตรวจสอบ"""
    m = len(score_list)
    w = np.ones(m) / m if weights is None else np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    f = minmax if scale == "minmax" else zscore
    return sum(wi * f(s) for wi, s in zip(w, score_list))


def rrf_fusion(score_list, k=60, weights=None):
    """Reciprocal Rank Fusion: Σ 1/(k + rank_i) ใช้เฉพาะ "อันดับ" จึงไม่ต้องปรับมาตรา (k=60 ตามธรรมเนียม)"""
    n = len(score_list[0])
    out = np.zeros(n)
    w = np.ones(len(score_list)) if weights is None else np.asarray(weights)
    for wi, s in zip(w, score_list):
        ranks = np.empty(n)
        ranks[np.argsort(-np.asarray(s), kind="stable")] = np.arange(1, n + 1)
        out += wi / (k + ranks)
    return out


class PlattCalibrator:
    """Platt scaling (นิยาม 3.19): p = σ(a·s + b) ประมาณ a, b จากชุดตรวจสอบด้วย log-loss ต่ำสุด
    (ใช้ LogisticRegression 1 มิติ) — ห้ามใช้ชุดทดสอบปรับเทียบ (รั่วไหลแบบที่ 5 ในตาราง 2.10)"""
    def fit(self, scores, labels):
        from sklearn.linear_model import LogisticRegression
        self.lr = LogisticRegression(C=1e6, max_iter=1000).fit(np.asarray(scores).reshape(-1, 1), labels)
        self.a_, self.b_ = float(self.lr.coef_[0, 0]), float(self.lr.intercept_[0])
        return self
    def predict_proba(self, scores):
        return self.lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]


def expected_calibration_error(probs, labels, n_bins=10):
    """ECE (นิยาม 3.20): แบ่งถังตามความมั่นใจ แล้วเฉลี่ย |สัดส่วนถูกจริง − ความมั่นใจเฉลี่ย| ถ่วงด้วยขนาดถัง
    คืน (ece, ตารางรายถัง) — ตัวอย่าง 3.23 ใช้ 5 ถัง"""
    probs, labels = np.asarray(probs), np.asarray(labels)
    edges = np.linspace(0, 1, n_bins + 1)
    rows, ece = [], 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (probs > lo) & (probs <= hi) if lo > 0 else (probs >= lo) & (probs <= hi)
        if m.sum() == 0:
            continue
        acc, conf = labels[m].mean(), probs[m].mean()
        ece += m.sum() / len(probs) * abs(acc - conf)
        rows.append({"ถัง": f"({lo:.1f}, {hi:.1f}]", "จำนวน": int(m.sum()), "สัดส่วนถูกจริง": round(acc, 4),
                     "ความมั่นใจเฉลี่ย": round(conf, 4), "|ผลต่าง|": round(abs(acc - conf), 4)})
    return ece, pd.DataFrame(rows)


# =========================================================================================
# ส่วนที่ 8  โมเดลทำนายคะแนน (Rating Prediction) — เพื่อวัด MAE/RMSE ของบทที่ 2 (2.1)
# =========================================================================================
class ContentKNNRegressor:
    """ทำนายคะแนน r̂(u,i) จากรายการที่ u เคยให้คะแนน ถ่วงด้วยความคล้าย "เชิงเนื้อหา" ระหว่างรายการ
        r̂(u,i) = μ_u + Σ_j sim(i,j)·(r_uj − μ_u) / Σ_j |sim(i,j)|     (j = k เพื่อนบ้านที่คล้ายที่สุดที่ u เคยให้คะแนน)
    * ไม่ต้องใช้ผู้ใช้คนอื่นเลย (ข้อดีข้อแรกของ content-based 3.19.1)
    * ถ้าไม่มีเพื่อนบ้าน (sim=0 ทุกคู่) จะถอยไปใช้ค่าเฉลี่ยของผู้ใช้ μ_u ; ถ้าผู้ใช้ใหม่ใช้ค่าเฉลี่ยรวม μ
    * shrink: ลดอิทธิพลเมื่อผลรวมความคล้ายเล็ก (กัน overfit เพื่อนบ้านน้อย)
    """
    def __init__(self, X, k=20, shrink=0.0, min_sim=0.0):
        self.Xn = l2_normalize_rows(X)
        self.k, self.shrink, self.min_sim = k, shrink, min_sim

    def fit(self, train):
        """train: DataFrame มีคอลัมน์ u (ดัชนีผู้ใช้), i (ดัชนีรายการ), r (คะแนน) — ต้องเป็น "ชุดฝึกเท่านั้น" """
        self.mu = float(train["r"].mean())
        self.user_hist = {u: (g["i"].to_numpy(), g["r"].to_numpy()) for u, g in train.groupby("u")}
        self.user_mean = {u: float(r.mean()) for u, (_, r) in self.user_hist.items()}
        return self

    def predict_one(self, u, i):
        if u not in self.user_hist:
            return self.mu
        items, ratings = self.user_hist[u]
        mu_u = self.user_mean[u]
        sims = np.asarray((self.Xn[items] @ self.Xn[i].T).todense()).ravel() if sp.issparse(self.Xn) \
            else self.Xn[items] @ self.Xn[i]
        m = sims > self.min_sim
        if m.sum() == 0:
            return mu_u
        sims, ratings = sims[m], ratings[m]
        if self.k and len(sims) > self.k:
            top = np.argsort(-sims)[: self.k]
            sims, ratings = sims[top], ratings[top]
        den = np.abs(sims).sum() + self.shrink
        return mu_u + float((sims * (ratings - mu_u)).sum() / den) if den > 0 else mu_u

    def predict(self, pairs):
        """pairs: DataFrame คอลัมน์ u, i → np.array คะแนนทำนาย (ผู้เรียกควร clip ให้อยู่ในช่วงคะแนนเอง)
        คำนวณเป็นกลุ่มต่อผู้ใช้ (เมทริกซ์ความคล้าย |ประวัติ| × |รายการที่ต้องทำนาย|) เร็วกว่าเรียก predict_one ทีละคู่มาก"""
        out = np.full(len(pairs), self.mu)
        pu, pi = pairs["u"].to_numpy(), pairs["i"].to_numpy()
        for u in np.unique(pu):
            m = np.where(pu == u)[0]
            if u not in self.user_hist:
                continue
            items, ratings = self.user_hist[u]; mu_u = self.user_mean[u]
            S = self.Xn[items] @ self.Xn[pi[m]].T
            S = np.asarray(S.todense()) if sp.issparse(S) else np.asarray(S)      # |hist| × |targets|
            S[S <= self.min_sim] = 0.0
            if self.k and S.shape[0] > self.k:                                   # เก็บเฉพาะ k เพื่อนบ้านที่คล้ายที่สุดต่อคอลัมน์
                thr = -np.partition(-S, self.k - 1, axis=0)[self.k - 1]
                S = np.where(S >= thr, S, 0.0)
            den = np.abs(S).sum(axis=0) + self.shrink
            num = (S * (ratings - mu_u)[:, None]).sum(axis=0)
            pred = np.where(den > 0, mu_u + num / np.where(den > 0, den, 1.0), mu_u)
            pred = np.where(np.abs(S).sum(axis=0) > 0, pred, mu_u)
            out[m] = pred
        return out


class MeanRatingBaseline:
    """ระบบฐานสำหรับ RMSE/MAE: 'global' = ค่าเฉลี่ยรวม, 'user' = ค่าเฉลี่ยผู้ใช้, 'item' = ค่าเฉลี่ยรายการ,
    'user_item' = μ + b_u + b_i (baseline ของ Koren) — ระบบใหม่ที่แพ้ค่าเฉลี่ยผู้ใช้คือสัญญาณเตือน"""
    def __init__(self, kind="user"):
        self.kind = kind
    def fit(self, train):
        self.mu = float(train["r"].mean())
        self.bu = (train.groupby("u")["r"].mean() - self.mu).to_dict()
        self.bi = (train.groupby("i")["r"].mean() - self.mu).to_dict()
        return self
    def predict(self, pairs):
        out = []
        for u, i in zip(pairs["u"], pairs["i"]):
            if self.kind == "global":
                out.append(self.mu)
            elif self.kind == "user":
                out.append(self.mu + self.bu.get(u, 0.0))
            elif self.kind == "item":
                out.append(self.mu + self.bi.get(i, 0.0))
            else:
                out.append(self.mu + self.bu.get(u, 0.0) + self.bi.get(i, 0.0))
        return np.array(out)


# =========================================================================================
# ส่วนที่ 9  ตัวชี้วัดของบทที่ 2
# =========================================================================================
# ---- 2.1 ตัวชี้วัดการทำนายคะแนน ----
def mae(y_true, y_pred):
    """Mean Absolute Error (2.1.1) — ทุกความผิดพลาดหนักเท่ากัน"""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true, y_pred):
    """Root Mean Squared Error (2.1.2) — ลงโทษความผิดพลาดใหญ่มากกว่า (ตัวอย่าง 2.2: MAE เท่ากันแต่ RMSE ต่างกัน)"""
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


# ---- 2.2–2.3 ตัวชี้วัดเชิงอันดับ "รายผู้ใช้" (rec_list = ลำดับรายการที่แนะนำ, rel = เซตรายการที่เกี่ยวข้อง) ----
def precision_at_k(rec, rel, K):
    """P@K = |Top-K ∩ Rel| / K  (2.2.1) ตัวหารคือ K เสมอ แม้ผู้ใช้จะมี Rel น้อยกว่า K"""
    return len(set(rec[:K]) & rel) / K


def recall_at_k(rec, rel, K):
    """R@K = |Top-K ∩ Rel| / |Rel|  (2.2.2) ; ผู้ใช้ที่ไม่มี Rel เลยควรถูกตัดออกก่อนประเมิน"""
    return len(set(rec[:K]) & rel) / len(rel) if rel else 0.0


def f1_at_k(rec, rel, K):
    p, r = precision_at_k(rec, rel, K), recall_at_k(rec, rel, K)
    return 2 * p * r / (p + r) if p + r > 0 else 0.0


def hit_at_k(rec, rel, K):
    """HR@K = 1 ถ้ามีรายการเกี่ยวข้องอย่างน้อยหนึ่งใน Top-K (2.2.4) — นิยมกับ leave-one-out"""
    return 1.0 if set(rec[:K]) & rel else 0.0


def rr_at_k(rec, rel, K):
    """Reciprocal Rank (2.3.1): 1/อันดับของรายการเกี่ยวข้อง "ตัวแรก" ; 0 ถ้าไม่มีใน K"""
    for pos, it in enumerate(rec[:K], start=1):
        if it in rel:
            return 1.0 / pos
    return 0.0


def ap_at_k(rec, rel, K):
    """Average Precision@K (2.3.2): เฉลี่ย P@i เฉพาะตำแหน่ง i ที่เป็นรายการเกี่ยวข้อง หารด้วย min(|Rel|, K)"""
    if not rel:
        return 0.0
    hits, s = 0, 0.0
    for pos, it in enumerate(rec[:K], start=1):
        if it in rel:
            hits += 1
            s += hits / pos
    return s / min(len(rel), K)


def dcg_at_k(gains):
    """DCG = Σ gain_i / log2(i+1) (2.3.3) — ตำแหน่ง 1 หารด้วย log2(2)=1, ตำแหน่ง 2 หารด้วย log2(3)…"""
    return float(sum(g / math.log2(i + 1) for i, g in enumerate(gains, start=1)))


def ndcg_at_k(rec, rel, K, rel_grades=None, gain_type="linear"):
    """nDCG@K = DCG@K / IDCG@K
    rel_grades : dict item→ระดับความเกี่ยวข้อง (เช่น rating) ถ้า None ใช้ 1/0 (binary)
    gain_type  : 'linear' gain=grade ; 'exp' gain=2^grade−1 (ต้องประกาศแบบที่ใช้ — ตาราง 2.16 ขั้น 4)"""
    def g(it):
        grade = (rel_grades.get(it, 0.0) if rel_grades else (1.0 if it in rel else 0.0))
        if it not in rel:
            grade = 0.0
        return (2 ** grade - 1) if gain_type == "exp" else grade
    gains = [g(it) for it in rec[:K]]
    ideal = sorted([g(it) for it in rel], reverse=True)[:K]
    idcg = dcg_at_k(ideal)
    return dcg_at_k(gains) / idcg if idcg > 0 else 0.0


def auc_user(scores, pos_idx, neg_idx):
    """AUC (นิยาม 2.1) นับคู่ทีละคู่: สัดส่วนคู่ (บวก, ลบ) ที่รายการบวกได้คะแนนสูงกว่า (เสมอนับ 0.5)"""
    if len(pos_idx) == 0 or len(neg_idx) == 0:
        return np.nan
    ps, ns = scores[list(pos_idx)], scores[list(neg_idx)]
    diff = ps[:, None] - ns[None, :]
    return float(((diff > 0).sum() + 0.5 * (diff == 0).sum()) / diff.size)


# ---- 2.5 ตัวชี้วัด Beyond-Accuracy (ระดับระบบ / ระดับรายการแนะนำ) ----
def catalog_coverage(rec_lists, n_items):
    """Coverage@K (2.5.1) = |⋃_u Top-K(u)| / |I| — รายการนับครั้งเดียวไม่ว่าจะโผล่กี่คน"""
    s = set()
    for r in rec_lists:
        s.update(r)
    return len(s) / n_items


def gini_exposure(rec_lists, n_items):
    """Gini ของการเปิดเผย (2.5.2): x_i = จำนวนครั้งที่รายการ i ถูกแนะนำ "เรียงจากน้อยไปมาก" (ขั้นบังคับ)
        G = Σ (2i − n − 1) x_i / (n Σ x_i)   ; 0 = เท่าเทียม, →1 = กระจุกที่รายการเดียว
    รายการที่ไม่เคยถูกแนะนำนับเป็น x=0 ด้วย (n = ทั้งคลัง)"""
    x = np.zeros(n_items)
    for r in rec_lists:
        for it in r:
            x[it] += 1
    x = np.sort(x)
    n = len(x)
    if x.sum() == 0:
        return 0.0
    i = np.arange(1, n + 1)
    return float(((2 * i - n - 1) * x).sum() / (n * x.sum()))


def head_items_from_train(train_item_counts, n_items, share=1 / 3):
    """เส้นแบ่งหัว–หาง (นิยาม 2.2, ตัวอย่าง 2.8): เรียงตามความนิยม "จากชุดฝึก" แล้วสะสมจนถึง share ของปฏิสัมพันธ์ทั้งหมด
    คืน set ของ head items (ที่เหลือ = long tail) — ต้องประกาศเกณฑ์ที่ใช้ทุกครั้ง และห้ามคำนวณจากข้อมูลทั้งหมด (รั่ว)"""
    counts = np.zeros(n_items)
    for i, c in train_item_counts.items():
        counts[i] = c
    order = np.argsort(-counts, kind="stable")
    cum, total, head = 0.0, counts.sum(), set()
    for i in order:
        if counts[i] == 0:
            break
        head.add(int(i)); cum += counts[i]
        if cum >= share * total:
            break
    return head


def long_tail_ratio(rec, head, K):
    """LongTailRatio@K = สัดส่วนรายการใน Top-K ที่ "ไม่ใช่" head"""
    top = rec[:K]
    return sum(1 for it in top if it not in head) / K if top else 0.0


def intra_list_diversity(rec, F, K):
    """ILD@K (นิยาม 2.3) = 2/(K(K−1)) Σ_{i<j} (1 − cos(f_i, f_j)) วัดจากคุณลักษณะรายการ F (เช่น เวกเตอร์แนว)
    ค่าเข้าใกล้ 1 = ต่างกันมาก ; รายการที่มีชิ้นเดียวคืน 0"""
    top = rec[:K]
    if len(top) < 2:
        return 0.0
    M = l2_normalize_rows(F[top])
    S = M @ M.T
    S = np.asarray(S.todense()) if sp.issparse(S) else np.asarray(S)
    n = len(top)
    iu = np.triu_indices(n, k=1)
    return float((1 - S[iu]).mean())


def novelty(rec, p_item, K):
    """Novelty@K (2.5.4) = (1/K) Σ −log2 p(i) ; p(i) = "สัดส่วนผู้ใช้" ในชุดฝึกที่เคยมีปฏิสัมพันธ์กับ i (หน่วยบิต)
    รายการที่ไม่มีใครเคยเห็นเลย p=0 → ใช้ p = 1/(จำนวนผู้ใช้+1) กัน log(0) (ต้องประกาศ)"""
    top = rec[:K]
    if not top:
        return 0.0
    return float(np.mean([-math.log2(max(p_item[it], p_item.min_p)) for it in top]))


class ItemPopularity:
    """เก็บ p(i) = สัดส่วนผู้ใช้ที่เคยมีปฏิสัมพันธ์กับ i (คำนวณจากชุดฝึกเท่านั้น) และจำนวนปฏิสัมพันธ์รายรายการ"""
    def __init__(self, train, n_items, n_users):
        c_users = train.groupby("i")["u"].nunique()
        self.p = np.zeros(n_items)
        self.p[c_users.index.to_numpy()] = c_users.to_numpy() / n_users
        self.count = train.groupby("i").size().to_dict()
        self.min_p = 1.0 / (n_users + 1)
    def __getitem__(self, i):
        return self.p[i]


def calibration_error(hist_items, rec, G, K):
    """Calibration (2.5.5): 0.5 Σ_c |q(c) − p(c)| ; p = สัดส่วนแนวในประวัติ, q = สัดส่วนแนวในรายการแนะนำ
    G = เมทริกซ์รายการ×แนว (0/1) ; ค่าใกล้ 0 = สอดคล้องกับประวัติ"""
    def dist(idx):
        if len(idx) == 0:
            return None
        v = np.asarray(G[list(idx)].sum(axis=0)).ravel().astype(float)
        return v / v.sum() if v.sum() > 0 else v
    p, q = dist(hist_items), dist(rec[:K])
    if p is None or q is None:
        return np.nan
    return float(0.5 * np.abs(q - p).sum())


def parity_gap(per_user_df, metric, group_col):
    """Fairness ฝั่งผู้ใช้ (2.5.6): ผลต่างของ Primary Metric ระหว่างกลุ่มผู้ใช้ (เช่น ประวัติยาว vs สั้น)"""
    g = per_user_df.groupby(group_col)[metric].mean()
    return float(g.max() - g.min()), g


# ---- ตัวประเมินหลัก: จัดอันดับ + คำนวณทุกตัวชี้วัดในรอบเดียว ----
def evaluate_ranking(score_fn, users, train_hist, test_rel, n_items, K=10, item_feat=None, pop=None,
                     head=None, genre_mat=None, rel_grades=None, candidate="full", m_neg=99, seed=SEED,
                     rerank_fn=None, min_score=None, item_ids=None, pool=None):
    """ประเมินโมเดลจัดอันดับหนึ่งตัวบนผู้ใช้ทุกคน คืน (ตารางรายผู้ใช้, สรุปค่าเฉลี่ย, dict รายการแนะนำ)

    score_fn(u)  → np.array คะแนนทุกรายการ (ดัชนี 0..n_items-1) ของผู้ใช้ u ; คืน None = ไม่มีหลักฐาน
    train_hist   : dict u → {item_index: rating} จากชุดฝึก (ใช้กรองที่เคยเห็น และ calibration)
    test_rel     : dict u → set(item_index) ที่เกี่ยวข้องในชุดที่กันไว้ (เช่น rating ≥ 4)
    candidate    : 'full' = จัดอันดับทั้งคลังที่ยังไม่เห็น (นิยาม 2.7) ; 'sampled' = เป้าหมาย + m_neg รายการลบสุ่ม
                   (ต้องรายงานว่าเป็น sampled metric และห้ามเทียบข้าม m — 2.9.5) ; รายการลบไม่ถูกวางไว้ตำแหน่งแรก
    rerank_fn(u, scores) → list ดัชนี (เช่น MMR) ใช้แทนการเรียงตามคะแนนเมื่อกำหนด
    item_feat    : เมทริกซ์คุณลักษณะสำหรับ ILD (เช่น เวกเตอร์แนว) ; genre_mat สำหรับ calibration
    pop          : ItemPopularity สำหรับ Novelty ; head : เซต head items สำหรับ LongTail
    pool         : เซตดัชนีรายการที่ "ให้บริการได้" (servable catalog เช่น รายการที่อยู่ใน k-core ของชุดฝึก)
                   None = ทั้งคลัง ; เมื่อกำหนด ทั้ง candidate, AUC และตัวหารของ Coverage/Gini จะใช้เซตนี้ (ต้องประกาศ)
    ผู้ใช้ที่ไม่มีรายการเกี่ยวข้องในชุดทดสอบจะถูกข้าม (ประกาศจำนวนที่ประเมินจริงเสมอ)
    """
    rng = np.random.default_rng(seed)
    rows, rec_lists, recs = [], [], {}
    pool_arr = np.arange(n_items) if pool is None else np.asarray(sorted(pool))
    pool_set = None if pool is None else set(int(i) for i in pool_arr)
    n_cat = len(pool_arr)
    for u in users:
        rel = test_rel.get(u, set())
        if not rel:
            continue
        seen = set(train_hist.get(u, {}).keys())
        s = score_fn(u)
        if s is None:
            s = np.zeros(n_items)
        s = np.asarray(s, dtype=np.float64)
        cands = pool_set
        if candidate == "sampled":
            unseen = np.setdiff1d(pool_arr, list(seen | rel))
            neg = rng.choice(unseen, size=min(m_neg, len(unseen)), replace=False)
            cands = set(neg.tolist()) | rel
        if rerank_fn is not None:
            rec = rerank_fn(u, s)
        else:
            rec = top_k(s, seen, K, min_score=min_score, item_ids=item_ids, candidates=cands)
        recs[u] = rec
        rec_lists.append(rec)
        row = {"user": u, "n_rel": len(rel), "n_hist": len(seen),
               "P@K": precision_at_k(rec, rel, K), "R@K": recall_at_k(rec, rel, K), "F1@K": f1_at_k(rec, rel, K),
               "HR@K": hit_at_k(rec, rel, K), "RR@K": rr_at_k(rec, rel, K), "AP@K": ap_at_k(rec, rel, K),
               "nDCG@K": ndcg_at_k(rec, rel, K, rel_grades.get(u) if rel_grades else None)}
        if candidate == "full":
            neg_pool = np.setdiff1d(pool_arr, list(seen | rel))
            if len(neg_pool) > 500:
                neg_pool = rng.choice(neg_pool, 500, replace=False)   # ประมาณ AUC ด้วยตัวอย่างลบ 500 ชิ้น
            row["AUC"] = auc_user(s, list(rel), neg_pool)
        if item_feat is not None:
            row["ILD@K"] = intra_list_diversity(rec, item_feat, K)
        if pop is not None:
            row["Novelty@K"] = novelty(rec, pop, K)
        if head is not None:
            row["LongTail@K"] = long_tail_ratio(rec, head, K)
        if genre_mat is not None:
            row["CalibErr"] = calibration_error(list(seen), rec, genre_mat, K)
        rows.append(row)
    per_user = pd.DataFrame(rows)
    summary = per_user.drop(columns=["user", "n_rel", "n_hist"]).mean(numeric_only=True).to_dict()
    summary["MRR@K"] = summary.pop("RR@K", 0.0)
    summary["MAP@K"] = summary.pop("AP@K", 0.0)
    summary["Coverage@K"] = catalog_coverage(rec_lists, n_cat)      # ตัวหาร = ขนาดคลังที่ให้บริการได้ (pool) หรือทั้งคลัง
    summary["Gini"] = gini_exposure(rec_lists, n_items) if pool is None else gini_exposure(
        [[int(np.searchsorted(pool_arr, it)) for it in r if it in pool_set] for r in rec_lists], n_cat)
    summary["n_users"] = len(per_user)
    return per_user, summary, recs


# =========================================================================================
# ส่วนที่ 10  การออกแบบการทดลองออฟไลน์ (หัวข้อ 2.7–2.10, 2.14)
# =========================================================================================
def encode_ids(df, user_col="userId", item_col="movieId", items_df=None, item_id_col="movieId"):
    """แปลงรหัสผู้ใช้/รายการเป็นดัชนี 0..n-1 (คอลัมน์ u, i) — รายการอ้างอิงตาม items_df เพื่อให้คลังครบทุกชิ้น
    แม้ไม่มีใครให้คะแนน (จำเป็นต่อ Coverage ที่ตัวหารคือ "ทั้งคลัง")"""
    users = sorted(df[user_col].unique())
    u_map = {u: k for k, u in enumerate(users)}
    if items_df is not None:
        items = list(items_df[item_id_col])
    else:
        items = sorted(df[item_col].unique())
    i_map = {it: k for k, it in enumerate(items)}
    out = df.copy()
    out["u"] = out[user_col].map(u_map)
    out["i"] = out[item_col].map(i_map)
    out = out.dropna(subset=["u", "i"]).astype({"u": int, "i": int})
    if "rating" in out.columns:
        out["r"] = out["rating"].astype(float)
    return out, u_map, i_map


def kcore_filter(df, k_user=5, k_item=5, max_iter=20):
    """กรอง k-core "ก่อน" แบ่งข้อมูล (รูปแบบรั่วไหลที่ 3 ในตาราง 2.10 คือกรองหลังแบ่ง) — วนจนนิ่ง"""
    d = df
    for _ in range(max_iter):
        n0 = len(d)
        uc = d.groupby("u").size(); ic = d.groupby("i").size()
        d = d[d["u"].isin(uc[uc >= k_user].index) & d["i"].isin(ic[ic >= k_item].index)]
        if len(d) == n0:
            break
    return d


def split_random(df, test_frac=0.2, seed=SEED):
    """แบบที่ 1 สุ่มโดยไม่สนเวลา — ง่ายแต่ "ใช้อนาคตทำนายอดีต" เมื่อข้อมูลมีเวลากำกับ (2.8.1)"""
    rng = np.random.default_rng(seed)
    m = rng.random(len(df)) < test_frac
    return df[~m].copy(), df[m].copy()


def split_global_temporal(df, test_frac=0.2, time_col="timestamp"):
    """แบบที่ 2 จุดตัดเวลาจุดเดียวทั้งระบบ: ก่อน = ฝึก หลัง = ทดสอบ (จำลองการใช้งานจริงตรงที่สุด)
    ข้อสังเกต: ผู้ใช้บางคนอาจไม่มีข้อมูลฝึกเลย (ผู้ใช้ใหม่จริง ๆ) — ต้องตัดสินใจว่าจะประเมินหรือไม่และประกาศ"""
    cut = df[time_col].quantile(1 - test_frac)
    return df[df[time_col] <= cut].copy(), df[df[time_col] > cut].copy()


def split_per_user_temporal(df, test_frac=0.2, time_col="timestamp", min_train=1):
    """แบบที่ 3 กันส่วนท้าย test_frac ของ "แต่ละคน" — ทุกคนมีชุดทดสอบ แต่จุดตัดไม่ตรงกัน (ยังรั่วข้ามผู้ใช้ได้)"""
    tr, te = [], []
    for u, g in df.sort_values(time_col).groupby("u"):
        n_te = int(np.ceil(len(g) * test_frac))
        n_te = min(n_te, max(0, len(g) - min_train))
        if n_te == 0:
            tr.append(g); continue
        tr.append(g.iloc[:-n_te]); te.append(g.iloc[-n_te:])
    return pd.concat(tr), (pd.concat(te) if te else df.iloc[0:0].copy())


def split_leave_one_out(df, time_col="timestamp", only_relevant=False, like_thr=4, seed=SEED):
    """แบบที่ 4 กันเหตุการณ์ "ล่าสุด" ของแต่ละคนไว้หนึ่งชิ้น (ไม่มีเวลา → สุ่มหนึ่งชิ้น)
    only_relevant=True: กันเฉพาะรายการที่ชอบ (rating ≥ like_thr) เพื่อให้ทุกผู้ใช้มีเป้าหมายวัด HR/MRR ได้
    ความแปรปรวนสูงมากเพราะตัดสินผู้ใช้ด้วยเหตุการณ์เดียว (2.8.1)"""
    rng = np.random.default_rng(seed)
    tr, te = [], []
    for u, g in df.groupby("u"):
        pool = g[g["r"] >= like_thr] if only_relevant and "r" in g else g
        if len(g) < 2 or len(pool) == 0:
            tr.append(g); continue
        if time_col in g.columns:
            hold = pool.sort_values(time_col).index[-1]
        else:
            hold = rng.choice(pool.index)
        te.append(g.loc[[hold]]); tr.append(g.drop(index=hold))
    return pd.concat(tr), pd.concat(te)


def three_way_split(df, protocol="global_temporal", valid_frac=0.1, test_frac=0.2, seed=SEED, **kw):
    """แบ่งสามชุด Train → Validation → Test (ตาราง 2.10 แถวที่ 5: เลือก hyperparameter บน validation เท่านั้น
    และแตะ test ครั้งเดียวตอนท้าย) โดยใช้โปรโตคอลเดียวกันสองครั้ง: ตัด test ออกก่อน แล้วตัด valid จากที่เหลือ"""
    f = {"random": split_random, "global_temporal": split_global_temporal,
         "per_user_temporal": split_per_user_temporal}[protocol]
    if protocol == "random":
        rest, test = f(df, test_frac, seed=seed)
        train, valid = f(rest, valid_frac / (1 - test_frac), seed=seed + 1)
    else:
        rest, test = f(df, test_frac, **kw)
        train, valid = f(rest, valid_frac / (1 - test_frac), **kw)
    return train, valid, test


def leakage_check(train, test, time_col="timestamp"):
    """ตรวจรั่วไหลเชิงเวลา (2.8.4): นับเหตุการณ์ในชุดฝึกที่เกิด "หลัง" เหตุการณ์แรกของชุดทดสอบ
    0 = ปลอดภัยแบบ global temporal ; >0 = มีการรั่วข้ามเวลา (ยอมรับได้เฉพาะเมื่อประกาศ เช่น per-user split)"""
    if time_col not in train.columns or len(test) == 0:
        return {"future_events_in_train": None}
    t0 = test[time_col].min()
    n_future = int((train[time_col] > t0).sum())
    return {"future_events_in_train": n_future, "share": n_future / max(1, len(train)),
            "train_max_time": pd.to_datetime(train[time_col].max(), unit="s"),
            "test_min_time": pd.to_datetime(t0, unit="s")}


def hist_from(df):
    """dict u → {i: r} จาก DataFrame (ใช้กับชุดฝึก)"""
    return {u: dict(zip(g["i"], g["r"])) for u, g in df.groupby("u")}


def relevant_from(df, like_thr=4.0):
    """dict u → set(i) ที่ rating ≥ like_thr (นิยามความเกี่ยวข้อง — ต้องประกาศ ขั้นที่ 4 ของโปรโตคอล)
    และ dict u → {i: r} สำหรับ nDCG แบบ graded"""
    rel, grades = {}, {}
    for u, g in df.groupby("u"):
        m = g["r"] >= like_thr
        rel[u] = set(g.loc[m, "i"])
        grades[u] = dict(zip(g.loc[m, "i"], g.loc[m, "r"]))
    return rel, grades


def paired_bootstrap(a, b, n_boot=2000, seed=SEED, alpha=0.05):
    """Paired bootstrap ระดับผู้ใช้ (2.10.2): สุ่มซ้ำ "ผู้ใช้" (ไม่ใช่ปฏิสัมพันธ์ — ตัวอย่าง 2.17) แล้วดู CI ของผลต่าง b−a
    คืน dict {diff, ci_low, ci_high, p_value(สองด้าน โดยประมาณ), significant}"""
    a, b = np.asarray(a, float), np.asarray(b, float)
    rng = np.random.default_rng(seed)
    n = len(a)
    d = b - a
    boots = np.array([d[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return {"diff": float(d.mean()), "ci_low": float(lo), "ci_high": float(hi), "p_value": float(min(1.0, p)),
            "significant": bool(lo > 0 or hi < 0), "n_users": n}


def holm_bonferroni(pvals, alpha=0.05):
    """Holm–Bonferroni (2.10.4): เรียง p จากน้อยไปมาก เทียบกับ α/(m−k+1) ทีละขั้น หยุดเมื่อไม่ผ่านครั้งแรก
    คืน list ของ bool (ปฏิเสธ H0 หรือไม่) ตามลำดับเดิม"""
    m = len(pvals)
    order = np.argsort(pvals)
    reject = [False] * m
    for k, idx in enumerate(order):
        if pvals[idx] <= alpha / (m - k):
            reject[idx] = True
        else:
            break
    return reject


def seed_variance(run_fn, seeds, metric="nDCG@K"):
    """ความผันผวนจาก seed อย่างเดียว (2.10.1): run_fn(seed) → summary dict ; คืนตาราง + mean±sd
    หากผลต่างระหว่างสองโมเดลเล็กกว่า sd ของ seed ก็ยังสรุปไม่ได้ว่าใครดีกว่า"""
    rows = [{"seed": s, **run_fn(s)} for s in seeds]
    df = pd.DataFrame(rows)
    return df, float(df[metric].mean()), float(df[metric].std(ddof=1)) if len(df) > 1 else 0.0


class ProtocolCard(dict):
    """Evaluation Protocol Card (นิยาม 2.9 / ตาราง 2.15–2.16): บันทึกทุกข้อกำหนดของการทดลอง "ก่อน" ดูผล
    ใช้เป็น dict ธรรมดา แล้วเรียก .show() เพื่อพิมพ์เป็นตาราง 11 ขั้น"""
    FIELDS = ["1 เป้าหมายผู้ใช้/ธุรกิจ", "2 Primary Metric", "3 Guardrail Metrics", "4 สูตรและค่า K / นิยามความเกี่ยวข้อง",
              "5 การแบ่งข้อมูล", "6 การป้องกันรั่วไหล", "7 Candidate Set", "8 Negative Sampling",
              "9 Accuracy + Beyond-Accuracy", "10 ความไม่แน่นอน", "11 ตรวจกับผู้ใช้จริง"]
    def show(self):
        rows = [{"ขั้น": f, "ค่าที่กำหนด": self.get(f, "—")} for f in self.FIELDS]
        for k, v in self.items():
            if k not in self.FIELDS:
                rows.append({"ขั้น": k, "ค่าที่กำหนด": v})
        df = pd.DataFrame(rows)
        try:
            from IPython.display import display
            display(df.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).hide(axis="index"))
        except Exception:
            print(df.to_string(index=False))


# =========================================================================================
# ส่วนที่ 11  การนำเสนอผล ตาราง กราฟ รายงานผู้ใช้ และฟอร์ม GUI
# =========================================================================================
ACC_COLS = ["P@K", "R@K", "F1@K", "HR@K", "MRR@K", "MAP@K", "nDCG@K", "AUC"]
BEYOND_COLS = ["Coverage@K", "Gini", "LongTail@K", "ILD@K", "Novelty@K", "CalibErr"]


def results_table(summaries, cols=None, K=None):
    """รวม summary dict ของหลายโมเดลเป็นตารางเดียว (แถว = โมเดล) ; แทน @K ด้วยค่า K จริงเมื่อระบุ"""
    df = pd.DataFrame(summaries).T
    cols = cols or [c for c in ACC_COLS + BEYOND_COLS + ["n_users"] if c in df.columns]
    df = df[cols]
    if "n_users" in df.columns:
        df["n_users"] = df["n_users"].astype(int)
    if K is not None:
        df.columns = [c.replace("@K", f"@{K}") for c in df.columns]
    return df


def show_df(df, caption=None, fmt="{:.4f}"):
    """แสดง DataFrame แบบสวยใน notebook (fallback เป็น print เมื่อไม่มี IPython)"""
    try:
        from IPython.display import display
        num_cols = [c for c in df.columns if pd.api.types.is_float_dtype(df[c])]
        sty = df.style.format(fmt, na_rep="—", subset=num_cols) if num_cols else df.style
        if caption:
            sty = sty.set_caption(caption)
        display(sty)
    except Exception:
        if caption:
            print(caption)
        print(df.to_string())


def plot_metric_bars(table, metrics, title="", ncols=4, figsize=None, highlight_best=True):
    """กราฟแท่งเทียบโมเดล: หนึ่งกราฟย่อยต่อหนึ่งตัวชี้วัด (ระบายแท่งที่ดีที่สุดเข้มกว่า)
    ตัวชี้วัดที่ "ยิ่งน้อยยิ่งดี" (Gini, CalibErr, MAE, RMSE) จะทำเครื่องหมาย ↓ ในชื่อ"""
    import matplotlib.pyplot as plt
    metrics = [m for m in metrics if m in table.columns]
    n = len(metrics)
    if n == 0:
        return
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize or (4.2 * ncols, 3.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    lower_better = {"Gini", "CalibErr", "MAE", "RMSE"}
    for ax, m in zip(axes, metrics):
        vals = table[m].astype(float)
        lb = any(m.startswith(x) for x in lower_better)
        best = vals.idxmin() if lb else vals.idxmax()
        colors = ["#2E6B4A" if (highlight_best and idx == best) else "#9DBFA9" for idx in table.index]
        ax.bar(range(len(vals)), vals.values, color=colors)
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(table.index, rotation=45, ha="right", fontsize=8)
        ax.set_title(m + (" ↓" if lb else " ↑"), fontsize=10)
        for k, v in enumerate(vals.values):
            if not np.isnan(v):
                ax.text(k, v, f"{v:.3f}", ha="center", va="bottom", fontsize=7)
        ax.grid(axis="y", alpha=0.3)
    for ax in axes[n:]:
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


def plot_tradeoff(table, x="nDCG@K", y="ILD@K", title="Accuracy เทียบ Beyond-Accuracy"):
    """แผนภาพกระจาย: แกน x = accuracy, แกน y = beyond-accuracy — ดู "การแลกเปลี่ยน" (3.10.5, 3.20.1 ข้อ 9)"""
    import matplotlib.pyplot as plt
    if x not in table.columns or y not in table.columns:
        return
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for name, row in table.iterrows():
        ax.scatter(row[x], row[y], s=60, color="#2E6B4A")
        ax.annotate(name, (row[x], row[y]), textcoords="offset points", xytext=(5, 4), fontsize=8)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_title(title); ax.grid(alpha=0.3)
    plt.show()


def side_by_side_topk(recs_by_model, u, item_title, K=5, scores_by_model=None):
    """ตารางเทียบรายการแนะนำ Top-K ของผู้ใช้ u จากหลายโมเดล (คอลัมน์ = โมเดล) — ตัวอย่างผลลัพธ์บนชุดตรวจสอบ"""
    cols = {}
    for name, recs in recs_by_model.items():
        r = recs.get(u, [])[:K]
        cells = []
        for it in r:
            t = item_title(it)
            if scores_by_model and name in scores_by_model and scores_by_model[name].get(u) is not None:
                t += f" ({scores_by_model[name][u][it]:.3f})"
            cells.append(t)
        cells += [""] * (K - len(cells))
        cols[name] = cells
    return pd.DataFrame(cols, index=[f"อันดับ {k}" for k in range(1, K + 1)])


def mark_hits(df_topk, rel_titles):
    """ใส่เครื่องหมาย ✔ หน้ารายการที่อยู่ในของจริง (ground truth) เพื่ออ่านผลด้วยตา"""
    def f(x):
        if not x:
            return x
        base = re.sub(r" \(-?\d+\.\d+\)$", "", x)      # ตัดเฉพาะคะแนนท้ายข้อความ ไม่ตัดวงเล็บในชื่อเรื่อง เช่น "Toy Story (1995)"
        return ("✔ " if base in rel_titles else "   ") + x
    return df_topk.applymap(f) if hasattr(df_topk, "applymap") else df_topk.map(f)


def user_report(u, train_hist, test_df, recs_by_model, item_title, item_content=None, K=5, like_thr=4):
    """รายงานผู้ใช้หนึ่งคน 3 ส่วน: ① ประวัติ (ชุดฝึก) ② ของจริงในชุดที่กันไว้ ③ Top-K ของแต่ละโมเดล (✔ = ตรงของจริง)
    item_content(i) → str เนื้อหาของรายการ (ถ้ามี) ; คืน dict ของ DataFrame"""
    hist = train_hist.get(u, {})
    h = pd.DataFrame([{"item": item_title(i), "rating": r, **({"เนื้อหา": item_content(i)} if item_content else {})}
                      for i, r in sorted(hist.items(), key=lambda x: -x[1])])
    te = test_df[test_df["u"] == u]
    rel_titles = set(item_title(i) for i in te.loc[te["r"] >= like_thr, "i"])
    g = pd.DataFrame([{"item": item_title(i), "rating": r, "เกี่ยวข้อง": "✔" if r >= like_thr else "",
                       **({"เนื้อหา": item_content(i)} if item_content else {})}
                      for i, r in zip(te["i"], te["r"])])
    top = mark_hits(side_by_side_topk(recs_by_model, u, item_title, K), rel_titles)
    return {"history": h, "ground_truth": g, "recommended": top}


def build_recommender_form(user_ids, model_names, recommend_fn, user_info_fn, K_default=5, K_max=20,
                           title="ตัวอย่างการประยุกต์ใช้: ฟอร์มแนะนำภาพยนตร์"):
    """ฟอร์ม GUI ด้วย ipywidgets (มีใน Colab อยู่แล้ว)
    recommend_fn(user_id, model_name, K) → DataFrame รายการที่แนะนำ (พร้อมเนื้อหา/คะแนน)
    user_info_fn(user_id)                → DataFrame ข้อมูล/ประวัติของผู้ใช้
    """
    try:
        import ipywidgets as W
        from IPython.display import display, HTML, clear_output
    except Exception:
        print("ไม่มี ipywidgets — ติดตั้งด้วย pip install ipywidgets")
        return None
    w_user = W.Dropdown(options=list(user_ids), description="ผู้ใช้", layout=W.Layout(width="260px"))
    w_model = W.Dropdown(options=list(model_names), description="โมเดล", layout=W.Layout(width="320px"))
    w_k = W.IntSlider(value=K_default, min=1, max=K_max, description="K", continuous_update=False)
    btn = W.Button(description="แนะนำ", button_style="success", icon="play")
    out = W.Output()

    def run(_=None):
        with out:
            clear_output()
            display(HTML(f"<h4 style='color:#2E6B4A'>ข้อมูลผู้ใช้ {w_user.value}</h4>"))
            display(user_info_fn(w_user.value))
            display(HTML(f"<h4 style='color:#2E6B4A'>รายการที่แนะนำ Top-{w_k.value} โดย {w_model.value}</h4>"))
            display(recommend_fn(w_user.value, w_model.value, w_k.value))
    btn.on_click(run)
    w_user.observe(lambda c: run() if c["name"] == "value" else None, names="value")
    w_model.observe(lambda c: run() if c["name"] == "value" else None, names="value")
    box = W.VBox([W.HTML(f"<h3 style='color:#2E6B4A'>{title}</h3>"), W.HBox([w_user, w_model, w_k, btn]), out])
    display(box)
    run()
    return box


## ส่วน 2 — โหลดชุดข้อมูล Goodbooks-10k และสร้างตารางหนังสือ

ชุดข้อมูล (zygmuntz/goodbooks-10k บน GitHub · ดาวน์โหลดได้โดยไม่ต้องล็อกอิน)
* `ratings.csv` — user_id, book_id, rating (1–5) **ไม่มี timestamp** · 5.98 ล้านแถว
* `books.csv` — title, authors, original_publication_year, language_code, average_rating, ratings_count, image_url
* `book_tags.csv` + `tags.csv` — tag ที่ผู้อ่านติดให้แต่ละเล่ม พร้อมจำนวนครั้ง (ใช้เป็น "เนื้อหา" หลักเหมือน tag ของ MovieLens)

**ข้อสังเกต** — เราสุ่ม *ผู้ใช้* (ไม่ใช่สุ่มแถว) เพื่อให้ประวัติแต่ละคนครบ และคลังสำหรับ Coverage คือหนังสือทั้ง 10,000 เล่ม

In [ ]:
def load_goodbooks(data_dir=DATA_DIR):
    base = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/"
    os.makedirs(data_dir, exist_ok=True)
    for f in ["books.csv", "tags.csv", "book_tags.csv", "ratings.csv"]:
        p = os.path.join(data_dir, f)
        if not os.path.exists(p):
            print("ดาวน์โหลด", f, "…"); urllib.request.urlretrieve(base + f, p)
    books = pd.read_csv(os.path.join(data_dir, "books.csv"))
    tags = pd.read_csv(os.path.join(data_dir, "tags.csv"))
    book_tags = pd.read_csv(os.path.join(data_dir, "book_tags.csv"))
    ratings = pd.read_csv(os.path.join(data_dir, "ratings.csv"))
    return books, tags, book_tags, ratings

books, tags, book_tags, ratings_all = load_goodbooks()
print(f"books {books.shape} · tags {tags.shape} · book_tags {book_tags.shape} · ratings {ratings_all.shape}")

# ---------- tag ต่อเล่ม: ตัด tag ทั่วไปที่ไม่บอกเนื้อหา แล้วเก็บ 15 tag ที่ถูกติดมากที่สุด ----------
GENERIC = {"to-read", "currently-reading", "favorites", "books-i-own", "owned", "owned-books", "library", "kindle", "ebook",
           "ebooks", "audiobook", "audiobooks", "my-books", "default", "re-read", "wish-list", "to-buy", "favourites", "series", "audio"}
bt = book_tags.merge(tags, on="tag_id")
bt["tag_name"] = bt["tag_name"].astype(str).str.lower().str.strip()
bt = bt[~bt["tag_name"].isin(GENERIC) & ~bt["tag_name"].str.match(r"^[\d\-]+$")]
bt = bt.sort_values(["goodreads_book_id", "count"], ascending=[True, False]).groupby("goodreads_book_id").head(15)
tag_text = bt.groupby("goodreads_book_id")["tag_name"].apply(lambda s: " ".join(t.replace("-", " ") for t in s)).rename("tag_text")
books = books.merge(tag_text, left_on="goodreads_book_id", right_index=True, how="left").fillna({"tag_text": ""})
def decade_token(y):
    """โทเคนทศวรรษที่เป็นตัวอักษรล้วน (ตัวตัดคำ [a-z]+ จะไม่แยกตัวเลข): 1990 → 'decadebjja' (b=1, j=9, a=0)"""
    if pd.isna(y) or y <= 0: return ""
    return "decade" + "".join("abcdefghij"[int(c)] for c in str(int(y) // 10 * 10))
books["decade_tok"] = books["original_publication_year"].apply(decade_token)
books["lang_tok"] = books["language_code"].fillna("").astype(str).str.replace("-", "").str.lower()
books = books.sort_values("book_id").reset_index(drop=True)       # book_id = 1..10000 → ดัชนี 0..9999
N_ITEMS = len(books)
def title_of(i):   return f"{books.title[i][:60]} — {books.authors[i][:30]}"
def content_of(i): return f"[{books.lang_tok[i]} {books.decade_tok[i]}] {books.tag_text[i][:70]}"
display(books[["book_id", "title", "authors", "original_publication_year", "language_code", "tag_text"]].head(5))

# ---------- สุ่มผู้ใช้ (ไม่สุ่มแถว) แล้วแปลงเป็นดัชนี u, i, r ----------
rng = np.random.default_rng(SEED)
uc = ratings_all.groupby("user_id").size()
elig = uc[uc >= MIN_RATINGS_USER].index.to_numpy()
keep_users = rng.choice(elig, size=N_USERS_SAMPLE, replace=False)
ratings = ratings_all[ratings_all.user_id.isin(keep_users)].copy()
df, u_map, i_map = encode_ids(ratings, "user_id", "book_id", items_df=books, item_id_col="book_id")
N_USERS = int(df.u.max()) + 1
print(f"ตัวอย่างที่ใช้: ผู้ใช้ {df.u.nunique()} · หนังสือที่มีคะแนน {df.i.nunique()} / {N_ITEMS} · คะแนน {len(df)} แถว · "
      f"ความหนาแน่น {len(df)/(N_USERS*N_ITEMS):.2%} · คะแนน ≥4 = {(df.r>=4).mean():.1%}")


## ส่วน 3 — เครื่องมือตรวจคำตอบ (รันก่อนเริ่มทำโจทย์)
`grade(...)` เก็บผลแต่ละข้อไว้สรุปท้ายเล่ม · ถ้าคำตอบยังเป็น `None` จะแจ้งว่า **ยังไม่ได้ทำ** ไม่ใช่ผิด

In [ ]:
SCORES = {}
def grade(task, checks):
    ok, msgs = True, []
    for label, cond in checks:
        try:
            passed = bool(cond() if callable(cond) else cond)
        except Exception as e:
            passed, label = False, f"{label}  [{type(e).__name__}: {str(e)[:60]}]"
        msgs.append(("  ✔ " if passed else "  ✘ ") + label); ok = ok and passed
    SCORES[task] = ok
    print(f"[{task}] {'ผ่าน ✅' if ok else 'ยังไม่ผ่าน ❌'}"); print("\n".join(msgs)); return ok

def not_done(*vals): return any(v is None for v in vals)

class TaskNotDone(Exception): pass

def need(**kw):
    missing = [k for k, v in kw.items() if v is None]
    if missing: raise TaskNotDone("ต้องทำข้อก่อนหน้าให้เสร็จก่อน — ยังไม่ได้เติม: " + ", ".join(missing))

# ประกาศตัวแปรผลลัพธ์ของทุกข้อไว้ล่วงหน้า เพื่อให้เซลล์ตรวจรันได้แม้ข้ามข้อ
for _v in ["TEXT", "VEC", "X", "idf_manual", "train", "valid", "test", "train_hist", "valid_rel", "valid_grades", "pop", "head",
           "POOL", "users_valid", "RESULTS", "RECS", "PU", "s_tfidf", "s_rocchio", "res_bm25", "BEST_K1", "BEST_B", "BM",
           "s_bm25", "Z", "s_hybrid", "cmp_cand", "tbl", "lam_choice", "tbl_test", "boot", "rep", "tbl_rating", "Q"]:
    globals().setdefault(_v, None)
RESULTS, RECS, PU = {}, {}, {}
print("พร้อมแล้ว")


## ★ E1 — สร้าง "เนื้อหา" ของหนังสือและเวกเตอร์ TF-IDF (หัวข้อ 3.4–3.6)

**โจทย์** สร้างเอกสารของหนังสือแต่ละเล่มจาก 5 เขตข้อมูล `title` · `authors` · `tag_text` · `lang_tok` · `decade_tok`
โดยให้ tag **ซ้ำ 2 ครั้ง** (เพิ่มน้ำหนักเขตข้อมูล — แนวคิด BM25F) แล้วสร้าง `TextVectorizer(tf="log", idf="smooth", min_df=2, max_df=0.85)`
และเมทริกซ์ `X` (L2 แล้ว)

*ข้อสังเกต*: ชื่อผู้แต่งเป็นคำเฉพาะ (IDF สูง) — ระบบจะ "แนะนำเล่มอื่นของผู้แต่งเดิม" ได้เองโดยไม่ต้องสอน

In [ ]:
def build_text(w_tag=2):
    doc = books["title"].fillna("") + " " + books["authors"].fillna("")
    for _ in range(w_tag):
        doc = doc + " " + books["tag_text"]
    doc = doc + " " + books["lang_tok"] + " " + books["decade_tok"]
    return list(doc.str.strip())

TEXT = None        # TODO: เรียก build_text(w_tag=2)
VEC = None         # TODO: TextVectorizer(tf="log", idf="smooth", min_df=2, max_df=0.85)
X = None           # TODO: VEC.fit_transform(TEXT)
if X is not None:
    print("X:", X.shape, "· ความหนาแน่น", f"{X.nnz/(X.shape[0]*X.shape[1]):.3%}", "· avgdl", round(VEC.avgdl_, 2))
    print("df สูงสุด 12 คำ:", sorted(VEC.df_.items(), key=lambda x: -x[1])[:12])
    q = 0
    print("คำเด่นของ", books.title[q], "→", VEC.top_terms(X[q], 8))


In [ ]:
grade("E1", [
    ("TEXT เป็น list ยาวเท่าจำนวนหนังสือ", lambda: isinstance(TEXT, list) and len(TEXT) == N_ITEMS),
    ("VEC ใช้ min_df=2 และ tf='log'", lambda: VEC.min_df == 2 and VEC.tf == "log"),
    ("X มีจำนวนแถว = จำนวนหนังสือ และคลังศัพท์ > 1,000 คำ", lambda: X.shape[0] == N_ITEMS and X.shape[1] > 1000),
    ("แถวของ X ผ่าน L2 (นอร์ม ≈ 1)", lambda: abs(np.sqrt(X[0].multiply(X[0]).sum()) - 1) < 1e-6),
])


## ★ E2 — คำนวณ IDF ด้วยมือแล้วเทียบกับที่ไลบรารีคำนวณ (สูตร smooth ของตำรา)

**โจทย์** ให้ `t = "romance"` (หรือคำใดก็ได้ที่อยู่ในคลังศัพท์) คำนวณ `idf_manual = ln((1+N)/(1+df_t)) + 1`
โดย N = จำนวนเอกสารทั้งหมด และ df_t = จำนวนเอกสารที่มีคำ t (ดูจาก `VEC.df_[t]`) แล้วเทียบกับ `VEC.idf_[VEC.idx_[t]]`

In [ ]:
need(VEC=VEC)
t = "romance"
N = VEC.N_; df_t = VEC.df_[t]
idf_manual = None      # TODO: ใช้สูตร np.log((1 + N) / (1 + df_t)) + 1
print(f"N={N} · df({t})={df_t} · idf_manual={idf_manual} · idf จากไลบรารี={VEC.idf_[VEC.idx_[t]]:.6f}")
# ลองเทียบกับคำที่พบน้อย เช่น ชื่อผู้แต่ง
for w in ["tolkien", "rowling", "fiction"]:
    if w in VEC.idx_: print(f"  idf({w}) = {VEC.idf_[VEC.idx_[w]]:.4f}  (df={VEC.df_[w]})")


In [ ]:
grade("E2", [("ยังไม่ได้ทำ", lambda: not not_done(idf_manual)),
             ("idf_manual ตรงกับไลบรารี (ต่างไม่เกิน 1e-9)", lambda: abs(idf_manual - VEC.idf_[VEC.idx_[t]]) < 1e-9)])


## ★★ E3 — แบ่งข้อมูลเมื่อ **ไม่มีเวลากำกับ** (หัวข้อ 2.8) และสถิติที่ต้องมาจากชุดฝึกเท่านั้น

Goodbooks ไม่มี timestamp → ใช้การแบ่งตามเวลาไม่ได้ ต้องใช้ `split_random` และ **ประกาศ** ข้อจำกัดนี้ (ตาราง 2.10 แถว 1)

**โจทย์**
1. `rest, test = split_random(df, 0.2, seed=SEED)` แล้ว `train, valid = split_random(rest, 0.125, seed=SEED+1)` (=10% ของทั้งหมด)
2. สร้าง `train_hist = hist_from(train)` · `valid_rel, valid_grades = relevant_from(valid, LIKE_THR)`
3. `pop = ItemPopularity(train, N_ITEMS, N_USERS)` · `head = head_items_from_train(pop.count, N_ITEMS)` · `POOL = set(train.i.unique())`
4. `users_valid` = ผู้ใช้ที่มีรายการเกี่ยวข้องใน valid **และ** มีประวัติใน train

In [ ]:
need(X=X)
rest, test = None, None            # TODO: split_random(df, 0.2, seed=SEED)
train, valid = None, None          # TODO: split_random(rest, 0.125, seed=SEED + 1)
train_hist = None                  # TODO: hist_from(train)
valid_rel, valid_grades = None, None   # TODO: relevant_from(valid, LIKE_THR)
pop = None                         # TODO: ItemPopularity(train, N_ITEMS, N_USERS)
head = None                        # TODO: head_items_from_train(pop.count, N_ITEMS)
POOL = None                        # TODO: set(int(i) for i in train.i.unique())
users_valid = None                 # TODO: sorted(u for u, s in valid_rel.items() if s and u in train_hist)
if train is not None:
    print(f"train {len(train)} · valid {len(valid)} · test {len(test)} · ผู้ใช้ประเมินได้ {len(users_valid)} · pool {len(POOL)} · head {len(head)}")
    assert not (set(zip(train.u, train.i)) & set(zip(test.u, test.i))), "คู่ (u,i) รั่วจาก train ไป test!"


In [ ]:
grade("E3", [
    ("แบ่งครบสามชุด", lambda: not not_done(train, valid, test)),
    ("สัดส่วน test ≈ 20% และ valid ≈ 10%", lambda: abs(len(test)/len(df) - 0.2) < 0.02 and abs(len(valid)/len(df) - 0.1) < 0.02),
    ("ไม่มีคู่ (u,i) ซ้ำข้ามชุด", lambda: not (set(zip(train.u, train.i)) & set(zip(valid.u, valid.i)) | set(zip(train.u, train.i)) & set(zip(test.u, test.i)))),
    ("pop คำนวณจาก train เท่านั้น (จำนวนปฏิสัมพันธ์รวม = len(train))", lambda: sum(pop.count.values()) == len(train)),
    ("users_valid ไม่ว่าง", lambda: len(users_valid) > 100),
])


## ★★ E4 — ระบบฐานที่ต้องมีเสมอ: TopPopular และ Random (หัวข้อ 2.2–2.5)

**โจทย์** เขียน `score_fn` สองตัว (รับ `u` คืน np.array ยาว N_ITEMS) แล้วประเมินด้วย `run_model(name, fn)` ที่ให้ไว้
* TopPopular → `pop.p.copy()` · Random → `rng.random(N_ITEMS)`

In [ ]:
need(users_valid=users_valid)
rng_m = np.random.default_rng(SEED)
pop_tb = 1e-6 * minmax(pop.p)          # ตัวตัดสินรองเมื่อคะแนนเสมอ = ความนิยม (3.10.2)

def run_model(name, fn, rerank_fn=None, users=None, hist=None, rel=None, grades=None, pool=None, candidate="full"):
    """ประเมินหนึ่งโมเดลด้วยโปรโตคอลเดียวกันทุกตัว แล้วเก็บผลไว้ใน RESULTS/RECS/PU"""
    pu, sm, recs = evaluate_ranking(fn, users or users_valid, hist or train_hist, rel or valid_rel, N_ITEMS, K=K, item_feat=X, pop=pop, head=head,
                                    rel_grades=grades or valid_grades, candidate=candidate, seed=SEED, rerank_fn=rerank_fn, pool=pool or POOL)
    RESULTS[name], RECS[name], PU[name] = sm, recs, pu
    print(f"{name:26s} nDCG@{K}={sm['nDCG@K']:.4f}  HR@{K}={sm['HR@K']:.4f}  Coverage@{K}={sm['Coverage@K']:.3f}  Novelty={sm['Novelty@K']:.2f}")
    return sm

def m_pop(u):    return None      # TODO: pop.p.copy()
def m_random(u): return None      # TODO: rng_m.random(N_ITEMS)
if m_pop(0) is not None:
    run_model("TopPopular", m_pop); run_model("Random", m_random)


In [ ]:
grade("E4", [("มีผลของ TopPopular และ Random", lambda: "TopPopular" in RESULTS and "Random" in RESULTS),
             ("TopPopular ชนะ Random ด้าน nDCG", lambda: RESULTS["TopPopular"]["nDCG@K"] > RESULTS["Random"]["nDCG@K"]),
             ("Random มี Coverage สูงกว่า TopPopular", lambda: RESULTS["Random"]["Coverage@K"] > RESULTS["TopPopular"]["Coverage@K"])])


## ★★★ E5 — โปรไฟล์ผู้ใช้ในปริภูมิ TF-IDF: centroid และ Rocchio (หัวข้อ 3.8–3.9)

**โจทย์** เขียน `s_tfidf(u)` = โคไซน์ระหว่างทุกเล่มกับโปรไฟล์ centroid ของ u และ `s_rocchio(u)` ด้วย β=0.8, γ=0.2
(ใช้ `build_profile(X, train_hist.get(u, {}), method, LIKE_THR, DISLIKE_THR, ...)` และ `match_scores(X, p)`) แล้วบวก `pop_tb` ก่อนคืน

In [ ]:
need(users_valid=users_valid)
def s_tfidf(u):
    return None    # TODO: match_scores(X, build_profile(X, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR)) + pop_tb
def s_rocchio(u):
    return None    # TODO: เหมือนข้างบนแต่ method="rocchio", beta=0.8, gamma=0.2
if s_tfidf(users_valid[0]) is not None:
    run_model("TFIDF-centroid", s_tfidf); run_model("TFIDF-Rocchio", s_rocchio)
    u0 = users_valid[0]; p0 = build_profile(X, train_hist[u0], "centroid", LIKE_THR, DISLIKE_THR)
    print("คำเด่นในโปรไฟล์ของผู้ใช้", u0, ":", VEC.top_terms(p0, 10))


In [ ]:
grade("E5", [("มีผลของ TFIDF-centroid และ TFIDF-Rocchio", lambda: "TFIDF-centroid" in RESULTS and "TFIDF-Rocchio" in RESULTS),
             ("โมเดลเนื้อหาชนะ Random อย่างชัดเจน (nDCG ≥ 3 เท่า)", lambda: RESULTS["TFIDF-centroid"]["nDCG@K"] >= 3 * RESULTS["Random"]["nDCG@K"]),
             ("Coverage ของโมเดลเนื้อหาสูงกว่า TopPopular", lambda: RESULTS["TFIDF-centroid"]["Coverage@K"] > RESULTS["TopPopular"]["Coverage@K"])])


## ★★★ E6 — BM25: ปรับ k₁ × b **บน validation เท่านั้น** (หัวข้อ 3.11.3 + ตาราง 2.10 แถว 5)

**โจทย์** ใช้ `BM25(k1, b).fit(VEC._train_tokens)` กับข้อคำถาม = คำเด่น 20 คำของโปรไฟล์ (ถ่วงด้วยน้ำหนัก) ลอง 4 จุด
`(k1, b) ∈ {(0.8, 0.0), (0.8, 0.75), (1.5, 0.0), (1.5, 0.75)}` บน `users_valid[:300]` (กลุ่มย่อยเพื่อประหยัดเวลา) เก็บผลใน `res_bm25`
แล้วเลือก `BEST_K1, BEST_B` จาก nDCG สูงสุด → สร้าง `BM` และ `s_bm25(u)` แล้วประเมินเต็มด้วย `run_model("BM25(tuned)", s_bm25)`

In [ ]:
need(users_valid=users_valid)
TOKENS = VEC._train_tokens
def bm25_scorer(bm, nq=20):
    def fn(u):
        p = build_profile(X, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR); top = VEC.top_terms(p, nq)
        return bm.score_query([t for t, _ in top], dict(top)) + pop_tb if top else None
    return fn
res_bm25 = []
for k1, b in [(0.8, 0.0), (0.8, 0.75), (1.5, 0.0), (1.5, 0.75)]:
    bm = None                          # TODO: BM25(k1=k1, b=b).fit(TOKENS)
    pu_, sm_, _ = None, None, None     # TODO: evaluate_ranking(bm25_scorer(bm), users_valid[:300], train_hist, valid_rel, N_ITEMS, K=K, rel_grades=valid_grades, seed=SEED, pool=POOL)
    res_bm25.append({"k1": k1, "b": b, "nDCG@10": sm_["nDCG@K"], "HR@10": sm_["HR@K"]}) if sm_ else None
res_bm25 = pd.DataFrame(res_bm25)
BEST_K1, BEST_B = None, None           # TODO: เลือกจากแถวที่ nDCG@10 สูงสุด
if BEST_K1 is not None:
    display(res_bm25); print("เลือก k1 =", BEST_K1, "b =", BEST_B)
    BM = BM25(k1=BEST_K1, b=BEST_B).fit(TOKENS); s_bm25 = bm25_scorer(BM)
    run_model("BM25(tuned)", s_bm25)


In [ ]:
grade("E6", [("มีตาราง res_bm25 ครบ 4 แถว", lambda: isinstance(res_bm25, pd.DataFrame) and len(res_bm25) == 4),
             ("BEST_K1/BEST_B ตรงกับแถว nDCG สูงสุด", lambda: (res_bm25.loc[res_bm25['nDCG@10'].idxmax(), ['k1', 'b']].tolist() == [BEST_K1, BEST_B])),
             ("มีผล BM25(tuned) ในตารางรวม", lambda: "BM25(tuned)" in RESULTS)])


## ★★★ E7 — ผสานคะแนนด้วย RRF: TF-IDF + BM25 + LSA (หัวข้อ 3.13.2, 3.18.1)

**โจทย์** สร้าง `Z = LSA(k=100, seed=SEED).fit(X).transform(X)` · `s_lsa(u)` แบบ centroid บน Z · แล้ว `s_hybrid(u) = rrf_fusion([s_tfidf(u), s_bm25(u), s_lsa(u)], k=60)`
ประเมินทั้ง `LSA(k=100)` และ `Hybrid-RRF`

In [ ]:
need(s_bm25=globals().get("s_bm25"))
Z = None                       # TODO: LSA(k=100, seed=SEED).fit(X).transform(X)
def s_lsa(u):    return None   # TODO: match_scores(Z, build_profile(Z, train_hist.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR)) + pop_tb
def s_hybrid(u): return None   # TODO: rrf_fusion([s_tfidf(u), s_bm25(u), s_lsa(u)], k=60) + pop_tb
if Z is not None:
    run_model("LSA(k=100)", s_lsa); run_model("Hybrid-RRF", s_hybrid)


In [ ]:
grade("E7", [("Z มี 100 มิติ", lambda: Z.shape == (N_ITEMS, 100)),
             ("มีผล LSA(k=100) และ Hybrid-RRF", lambda: "LSA(k=100)" in RESULTS and "Hybrid-RRF" in RESULTS),
             ("Hybrid ไม่แย่กว่า 0.9 เท่าของสมาชิกที่แย่ที่สุด", lambda: RESULTS["Hybrid-RRF"]["nDCG@K"] >= 0.9 * min(RESULTS[m]["nDCG@K"] for m in ["TFIDF-centroid", "BM25(tuned)", "LSA(k=100)"]))])


## ★★★★ E8 — Candidate set: full catalog เทียบ sampled negatives m=99 (หัวข้อ 2.9)

**โจทย์** ประเมิน `TFIDF-centroid` และ `TopPopular` อีกครั้งด้วย `candidate="sampled"` (ใช้ `run_model(..., candidate="sampled")` ตั้งชื่อลงท้าย `[sampled]`)
แล้วสร้าง `cmp_cand` = DataFrame เทียบ nDCG@10 และ HR@10 ของทั้งสองแบบ

In [ ]:
need(s_tfidf=s_tfidf)
# TODO: run_model("TFIDF-centroid [sampled]", s_tfidf, candidate="sampled") และ run_model("TopPopular [sampled]", m_pop, candidate="sampled")
cmp_cand = None   # TODO: pd.DataFrame({...}) แถว = โมเดล, คอลัมน์ = full nDCG, sampled nDCG, full HR, sampled HR
if cmp_cand is not None: display(cmp_cand.round(4))


In [ ]:
grade("E8", [("มี cmp_cand", lambda: isinstance(cmp_cand, pd.DataFrame) and len(cmp_cand) == 2),
             ("sampled สูงกว่า full ทั้งสองโมเดล", lambda: (cmp_cand["sampled(m=99) nDCG@10"] > cmp_cand["full nDCG@10"]).all())])


## ★★★★ E9 — ตารางรวม + Guardrail + MMR (หัวข้อ 2.5–2.6, 3.10.5)

**โจทย์**
1. `tbl = results_table({m: RESULTS[m] for m in RESULTS if "[sampled]" not in m}, K=K)` แล้วแสดงด้วย `show_df` และวาด `plot_metric_bars`
2. สร้าง `make_mmr(lam)` ที่จัดอันดับซ้ำ Top-50 ของ `s_hybrid` ด้วยความคล้ายในปริภูมิ `Z` แล้วประเมิน `λ ∈ {1.0, 0.7, 0.5}` (ตั้งชื่อ `Hybrid+MMR(λ=…)`)
3. `lam_choice` = λ ที่ ILD@10 สูงสุด **โดย nDCG@10 ไม่ต่ำกว่า 95% ของ λ=1.0** (guardrail ที่ประกาศ)

In [ ]:
need(s_hybrid=globals().get("s_hybrid"))
def make_mmr(lam):
    def rerank(u, s):
        base = top_k(s, set(train_hist.get(u, {})), K=50, candidates=POOL)
        if not base: return []
        Sm = np.asarray(Z[base] @ Z[base].T)
        return [base[j] for j in mmr_rerank(s[base], Sm, K=K, lam=lam, pool=len(base))]
    return rerank
for lam in [1.0, 0.7, 0.5]:
    pass   # TODO: run_model(f"Hybrid+MMR(λ={lam})", s_hybrid, rerank_fn=make_mmr(lam))
tbl = None          # TODO: results_table({m: RESULTS[m] for m in RESULTS if "[sampled]" not in m}, K=K)
lam_choice = None   # TODO: เลือกตามกฎ guardrail
if tbl is not None:
    show_df(tbl, f"ผลบน Validation (ผู้ใช้ {len(users_valid)} คน, K={K}, pool {len(POOL)} เล่ม)")
    plot_metric_bars(tbl, [f"nDCG@{K}", f"HR@{K}", f"Coverage@{K}", f"ILD@{K}", f"Novelty@{K}", "Gini"], title="Validation", ncols=3)
    print("Coverage ต่ำสุด:", tbl[f"Coverage@{K}"].idxmin(), "· nDCG สูงสุด:", tbl[f"nDCG@{K}"].idxmax(), "· λ ที่เลือก:", lam_choice)


In [ ]:
grade("E9", [("tbl มีทุกโมเดล (≥ 9 แถว)", lambda: isinstance(tbl, pd.DataFrame) and len(tbl) >= 9),
             ("มีผล MMR ครบ 3 ค่า λ", lambda: all(f"Hybrid+MMR(λ={l})" in RESULTS for l in [1.0, 0.7, 0.5])),
             ("lam_choice เป็นหนึ่งในสามค่าและผ่าน guardrail", lambda: lam_choice in (1.0, 0.7, 0.5) and RESULTS[f"Hybrid+MMR(λ={lam_choice})"]["nDCG@K"] >= 0.95 * RESULTS["Hybrid+MMR(λ=1.0)"]["nDCG@K"])])


## ★★★★ E10 — ทดสอบครั้งเดียวบน Test + ความไม่แน่นอน (หัวข้อ 2.10, 2.14)

**โจทย์**
1. `trainval = pd.concat([train, valid])` → `hist_tv = hist_from(trainval)` · `pop_tv` · `head_tv` · `POOL_TV` · `test_rel, test_grades` · `users_test`
2. ประเมิน 3 โมเดลบน test ด้วย `run_model(name + " [test]", fn, users=users_test, hist=hist_tv, rel=test_rel, grades=test_grades, pool=POOL_TV)`:
   `TopPopular` (ใช้ `pop_tv.p`) · `TFIDF-centroid` (โปรไฟล์จาก `hist_tv`) · `Hybrid-RRF` (สมาชิกทุกตัวใช้ `hist_tv`)
3. `boot = paired_bootstrap(a, b)` ของ nDCG@K รายผู้ใช้ ระหว่าง TopPopular (a) กับ Hybrid (b) บน test — ต่างอย่างมีนัยสำคัญหรือไม่

In [ ]:
need(tbl=tbl)
trainval = pd.concat([train, valid]); hist_tv = hist_from(trainval)
pop_tv = ItemPopularity(trainval, N_ITEMS, N_USERS); head_tv = head_items_from_train(pop_tv.count, N_ITEMS); POOL_TV = set(int(i) for i in trainval.i.unique())
test_rel, test_grades = relevant_from(test, LIKE_THR); users_test = sorted(u for u, s in test_rel.items() if s and u in hist_tv)
pop, head = pop_tv, head_tv          # ให้ตัวประเมินใช้สถิติจาก train+valid
pop_tb2 = 1e-6 * minmax(pop_tv.p)
def t_pop(u):    return pop_tv.p.copy()
def t_tfidf(u):  return match_scores(X, build_profile(X, hist_tv.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR)) + pop_tb2
def t_bm25(u):
    p = build_profile(X, hist_tv.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR); top = VEC.top_terms(p, 20)
    return BM.score_query([t for t, _ in top], dict(top)) + pop_tb2 if top else np.zeros(N_ITEMS)
def t_lsa(u):    return match_scores(Z, build_profile(Z, hist_tv.get(u, {}), "centroid", LIKE_THR, DISLIKE_THR)) + pop_tb2
def t_hybrid(u): return rrf_fusion([t_tfidf(u), t_bm25(u), t_lsa(u)], k=60) + pop_tb2
# TODO: run_model 3 ตัวบน test (ชื่อ "TopPopular [test]", "TFIDF-centroid [test]", "Hybrid-RRF [test]")
tbl_test = None     # TODO: results_table({m: RESULTS[m] for m in RESULTS if m.endswith("[test]")}, K=K)
boot = None         # TODO: paired_bootstrap(a, b) โดย a, b = nDCG@K รายผู้ใช้ (เรียงผู้ใช้ให้ตรงกันด้วย set_index("user"))
if tbl_test is not None:
    show_df(tbl_test, f"ผลบนชุดทดสอบ (ผู้ใช้ {len(users_test)} คน) — รายงานครั้งเดียว")
    print("paired bootstrap Hybrid − TopPopular (nDCG@10):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in boot.items()})
    pv = [paired_bootstrap(a.values, PU[m].set_index("user")["nDCG@K"].reindex(a.index).values, 1000, SEED)["p_value"] for m in ["TFIDF-centroid [test]", "Hybrid-RRF [test]"]]
    print("Holm–Bonferroni ปฏิเสธ H0:", dict(zip(["TFIDF", "Hybrid"], holm_bonferroni(pv))))


In [ ]:
grade("E10", [("tbl_test มี 3 แถว", lambda: isinstance(tbl_test, pd.DataFrame) and len(tbl_test) == 3),
              ("boot มี diff และ CI", lambda: isinstance(boot, dict) and all(k in boot for k in ["diff", "ci_low", "ci_high", "p_value"])),
              ("CI ล่าง ≤ diff ≤ CI บน", lambda: boot["ci_low"] <= boot["diff"] <= boot["ci_high"])])


## ☆ E11 — อ่านผลรายคน และทำนายคะแนน (MAE/RMSE)

**โจทย์**
1. `rep = user_report(u_show, hist_tv, test, {ชื่อ: RECS[ชื่อ] for ชื่อ ที่ลงท้าย [test]}, title_of, content_of, K=K, like_thr=LIKE_THR)` (`u_show` = ผู้ใช้ที่ Hybrid ทำได้ดีที่สุด กำหนดให้แล้ว)
   แล้วแสดง `rep["history"]`, `rep["ground_truth"]`, `rep["recommended"]` — นับว่าโมเดลใดมี ✔ มากที่สุด
2. `tbl_rating` = MAE/RMSE บน test ของ `MeanRatingBaseline("user")`, `MeanRatingBaseline("user_item")`, `ContentKNNRegressor(X, k=20)` (fit ด้วย `trainval`, clip 1–5)

In [ ]:
need(tbl_test=tbl_test)
u_show = int(PU["Hybrid-RRF [test]"].sort_values("nDCG@K", ascending=False).iloc[0]["user"])   # ผู้ใช้ที่ Hybrid ทำได้ดีที่สุด (ตัวอย่างที่อ่านง่าย) — ลองเปลี่ยนเป็น users_test[0] เพื่อดูกรณีที่พลาด
rep = None          # TODO: user_report(...)
tbl_rating = None   # TODO: pd.DataFrame({ชื่อ: {"MAE": mae(test.r, p), "RMSE": rmse(test.r, p)} ...}).T
if rep is not None:
    print(f"ผู้ใช้ {u_show} · ประวัติ {len(hist_tv[u_show])} เล่ม · ของจริง ≥4 ในชุดทดสอบ {len(test_rel[u_show])} เล่ม")
    show_df(rep["history"].head(8), "① ประวัติ (train+valid)", fmt="{}"); show_df(rep["ground_truth"].head(8), "② ของจริงในชุดทดสอบ", fmt="{}")
    show_df(rep["recommended"], "③ Top-10 ของแต่ละโมเดล (✔ = ตรงของจริง)", fmt="{}")
    hits = {c: rep["recommended"][c].str.startswith("✔").sum() for c in rep["recommended"].columns}; print("จำนวน ✔ ต่อโมเดล:", hits)
    show_df(tbl_rating, "MAE / RMSE บนชุดทดสอบ (ยิ่งน้อยยิ่งดี)")


In [ ]:
grade("E11", [("rep มีสามตาราง", lambda: isinstance(rep, dict) and all(k in rep for k in ["history", "ground_truth", "recommended"])),
              ("tbl_rating มี 3 โมเดล และ RMSE ≥ MAE ทุกแถว", lambda: len(tbl_rating) == 3 and (tbl_rating["RMSE"] >= tbl_rating["MAE"]).all())])


## ☆ Quiz — คำถามแนวคิด 10 ข้อ (ตรวจอัตโนมัติด้วย md5 ของเฉลย)

**1. ข้อใดอธิบายเหตุผลที่ต้องปรับ L2 ทั้งเวกเตอร์รายการและโปรไฟล์**  
a) เพื่อให้ค่าทุกมิติเป็นบวก  
b) เพื่อให้ผลคูณจุดเท่ากับโคไซน์และเอกสารยาวไม่ได้เปรียบ  
c) เพื่อลดขนาดคลังศัพท์

**2. Goodbooks ไม่มี timestamp ผลของการแบ่งแบบสุ่มจะ**  
a) ต่ำกว่าการใช้งานจริง  
b) เท่ากับการใช้งานจริง  
c) มักดีเกินจริงและต้องประกาศข้อจำกัด

**3. TopPopular มี Coverage@10 ต่ำมากเพราะ**  
a) แนะนำรายการชุดเดิมให้ผู้ใช้ทุกคน  
b) คะแนนความนิยมคำนวณผิด  
c) K เล็กเกินไป

**4. sampled nDCG@10 (m=99) สูงกว่า full เพราะ**  
a) โมเดลแม่นขึ้นเมื่อผู้สมัครน้อย  
b) รายการเป้าหมายมีคู่แข่งน้อยลงหลายสิบเท่า  
c) IDF เปลี่ยน

**5. ในสูตร BM25 พารามิเตอร์ b ควบคุม**  
a) การอิ่มตัวของความถี่  
b) การปรับตามความยาวเอกสาร  
c) จำนวนคำในข้อคำถาม

**6. RRF ไม่ต้องปรับมาตราคะแนนก่อนผสานเพราะ**  
a) ใช้เฉพาะอันดับ  
b) คะแนนทุกระบบอยู่ในช่วง 0–1 อยู่แล้ว  
c) มีการหารด้วยค่าสูงสุด

**7. กฎของ Rocchio ที่ตำราเน้นคือ**  
a) γ > β  
b) β > γ  
c) α ต้องเป็น 1 เสมอ

**8. MMR ที่ λ ต่ำลงมักทำให้**  
a) nDCG เพิ่มและ ILD เพิ่ม  
b) ILD เพิ่มแต่ nDCG อาจลด  
c) Coverage ลดลงเสมอ

**9. paired bootstrap ระดับผู้ใช้ต้องสุ่มซ้ำ**  
a) ปฏิสัมพันธ์ทีละแถว  
b) ผู้ใช้ทั้งคน  
c) รายการ

**10. การเลือก k1, b ของ BM25 ควรทำบน**  
a) test  
b) validation  
c) train

In [ ]:
import hashlib, json
Q = {1: "?", 2: "?", 3: "?", 4: "?", 5: "?", 6: "?", 7: "?", 8: "?", 9: "?", 10: "?"}   # TODO: เติม a/b/c
_KEY = "705285d20affe9bf290eded2d81fc6a9"
ok = hashlib.md5(json.dumps({int(k): v for k, v in Q.items()}, sort_keys=True).encode()).hexdigest() == _KEY
grade("Quiz", [("คำตอบครบ 10 ข้อ", lambda: all(v in ("a", "b", "c") for v in Q.values())), ("ตรงกับเฉลยทั้ง 10 ข้อ", lambda: ok)])


## สรุปคะแนน

In [ ]:
def score_report():
    n_ok = sum(SCORES.values()); n = len(SCORES)
    print(f"ผ่าน {n_ok} / {n} ข้อ")
    for k, v in SCORES.items():
        print(f"  {'✅' if v else '❌'} {k}")
score_report()
